# Backtest Service Research Engine Prototype

This notebook is the canonical algorithm and benchmark prototype for the future artifact-backed Backtest Service.

Scope:

- exchange/market/symbol artifact runtime, currently validated on `binance/spot/BTCUSDT`;
- signal timeframe: `15m`;
- artifact model: precomputed `.npy` prices, mappings, signals, and `hit_times/15m`;
- request model: indicator id + source set + window range;
- indicator arity benchmark: 1..7 indicators;
- execution modes:
  - no-risk;
  - risk-on TP/SL grid backed by `hit_times/15m`;
- benchmark output must capture warmup, runtime stages, memory, CPU and correctness/parity evidence.

This notebook is not tied to a historical persisted run id. It defines the target compute behavior that production service implementations must match.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import heapq
import itertools
import json
import math
import os
import platform
import resource
import threading
import time

import numba as nb
import numpy as np
import yaml

REQUEST_COORDINATES = {
    "exchange": "binance",
    "market_type": "spot",
    "symbol": "BTCUSDT",
}
TIMEFRAME = "15m"
REQUEST_PERIOD = {
    "start": datetime(2020, 1, 11, 20, 8, tzinfo=timezone.utc),
    "end": datetime(2026, 4, 11, 20, 8, tzinfo=timezone.utc),
    "semantics": "[start, end)",
}
REQUEST_INDICATORS = [
    {"indicator_id": "ma.dema", "sources": ["close"], "window_range": [5, 200]},
    {"indicator_id": "ma.hma", "sources": ["close"], "window_range": [5, 200]},
    {"indicator_id": "ma.ema", "sources": ["close"], "window_range": [5, 200]},
    {"indicator_id": "ma.sma", "sources": ["close"], "window_range": [5, 200]},
    {"indicator_id": "ma.wma", "sources": ["close"], "window_range": [5, 200]},
    {"indicator_id": "ma.rma", "sources": ["close"], "window_range": [5, 200]},
    {"indicator_id": "ma.zlema", "sources": ["close"], "window_range": [5, 200]},
]
REQUEST_RISK_NONE = {"mode": "none"}
REQUEST_RISK_TP_SL_GRID = {
    "mode": "tp_sl_grid",
    "timeframe": "15m",
    "tp_values_pct": np.arange(2.0, 25.0 + 0.5, 0.5).round(6).tolist(),
    "sl_values_pct": np.arange(2.0, 25.0 + 0.5, 0.5).round(6).tolist(),
}
REQUEST_EXECUTION = {
    "fee_rate": 0.00075,
    "slippage_rate": 0.0001,
    "initial_cash_quote": 10000.0,
    "sizing": {"mode": "all_in", "fixed_quote": 100.0},
    "profit_lock": {"enabled": False, "safe_profit_percent": 30.0},
    "direction_mode": "long_short_reversal",
    "close_on_end": True,
}
BACKTEST_REQUEST = {
    "coordinates": REQUEST_COORDINATES,
    "timeframe": TIMEFRAME,
    "period": REQUEST_PERIOD,
    "indicators": REQUEST_INDICATORS,
    "risk": REQUEST_RISK_NONE,
    "execution": REQUEST_EXECUTION,
    "top_n": 100,
    "sort_metric": "total_return_pct",
}

ARTIFACT_SLOT = os.environ.get("BACKTEST_ARTIFACT_SLOT", "slot_a")
ARTIFACT_ROOT = Path(os.environ.get(
    "BACKTEST_ARTIFACT_ROOT",
    f"/opt/roehub/state/backtest_artifacts/v2/{REQUEST_COORDINATES['exchange']}/{REQUEST_COORDINATES['market_type']}/{REQUEST_COORDINATES['symbol']}/{ARTIFACT_SLOT}",
))
PRICE_DIR_15M = ARTIFACT_ROOT / "prices" / TIMEFRAME
PRICE_DIR_1M = ARTIFACT_ROOT / "prices" / "1m"
MAPPINGS_15M_DIR = ARTIFACT_ROOT / "mappings" / TIMEFRAME
HIT_TIMES_DIR_15M = ARTIFACT_ROOT / "hit_times" / TIMEFRAME

COMMON_PRICE_SOURCES = ["close", "hlc3", "ohlc4", "low", "high", "open"]
INDICATOR_CATALOG_15M = {
    indicator_id: {
        "indicator_id": indicator_id,
        "sources": COMMON_PRICE_SOURCES,
        "param_name": "window",
        "param_values": list(range(5, 201)),
    }
    for indicator_id in (
        "ma.dema",
        "ma.hma",
        "ma.ema",
        "ma.sma",
        "ma.wma",
        "ma.rma",
        "ma.zlema",
        "ma.lwma",
        "ma.tema",
    )
}
INDICATOR_CATALOG_15M["ma.vwma"] = {
    "indicator_id": "ma.vwma",
    "sources": ["close"],
    "param_name": "window",
    "param_values": list(range(5, 201)),
}

INDICATOR_ARITY_SETS = {
    1: ("ma.dema",),
    2: ("ma.dema", "ma.hma"),
    3: ("ma.dema", "ma.hma", "ma.ema"),
    4: ("ma.dema", "ma.hma", "ma.ema", "ma.sma"),
    5: ("ma.dema", "ma.hma", "ma.ema", "ma.sma", "ma.wma"),
    6: ("ma.dema", "ma.hma", "ma.ema", "ma.sma", "ma.wma", "ma.rma"),
    7: ("ma.dema", "ma.hma", "ma.ema", "ma.sma", "ma.wma", "ma.rma", "ma.zlema"),
}
INDICATOR_ARITIES = tuple(INDICATOR_ARITY_SETS)

SLOT_MANIFEST_PATH = ARTIFACT_ROOT / "manifest.yaml"
PRICE_OPEN_TIME_15M_PATH = PRICE_DIR_15M / "open_time.i64.npy"
PRICE_CLOSE_TIME_15M_PATH = PRICE_DIR_15M / "close_time.i64.npy"
PRICE_OHLCV_15M_PATH = PRICE_DIR_15M / "ohlcv.f32.npy"
PRICE_OPEN_TIME_1M_PATH = PRICE_DIR_1M / "open_time.i64.npy"
PRICE_CLOSE_TIME_1M_PATH = PRICE_DIR_1M / "close_time.i64.npy"
PRICE_OHLCV_1M_PATH = PRICE_DIR_1M / "ohlcv.f32.npy"
BAR_OPEN_1M_IDX_15M_PATH = MAPPINGS_15M_DIR / "bar_open_1m_idx.u32.npy"
BAR_CLOSE_1M_IDX_15M_PATH = MAPPINGS_15M_DIR / "bar_close_1m_idx.u32.npy"
HIT_TIMES_MANIFEST_15M_PATH = HIT_TIMES_DIR_15M / "manifest.yaml"
TP_VALUES_15M_PATH = HIT_TIMES_DIR_15M / "tp_values.f32.npy"
SL_VALUES_15M_PATH = HIT_TIMES_DIR_15M / "sl_values.f32.npy"
LONG_TP_15M_PATH = HIT_TIMES_DIR_15M / "long_tp.u32.npy"
LONG_SL_15M_PATH = HIT_TIMES_DIR_15M / "long_sl.u32.npy"
SHORT_TP_15M_PATH = HIT_TIMES_DIR_15M / "short_tp.u32.npy"
SHORT_SL_15M_PATH = HIT_TIMES_DIR_15M / "short_sl.u32.npy"

USE_MMAP = True
ROW_PREFILTER_ROWS_PER_INDICATOR = 142
PREFILTER_TOP_FRAC = ROW_PREFILTER_ROWS_PER_INDICATOR / 588.0
PREFILTER_MIN_NONZERO = 1
COMBO_PREFILTER_TOP_FRAC = 1.0
COMBO_MIN_CONFIRM = 1
TIME_CHUNK = 4096
COMBO_CHUNK_SIZE = 4096
TOP_K_DEFAULT = int(BACKTEST_REQUEST["top_n"])
SELF_CHECK_N_DEFAULT = 8
BENCHMARK_ROWS_PER_INDICATOR = int(os.environ.get("BACKTEST_ENGINE_BENCHMARK_ROWS", "6"))
BENCHMARK_OUTPUT_ROOT = Path("docs/architecture/backtest/benchmark_iterations")
FEE_RATE = float(REQUEST_EXECUTION["fee_rate"])
SLIPPAGE_RATE = float(REQUEST_EXECUTION["slippage_rate"])
INIT_CASH_QUOTE = float(REQUEST_EXECUTION["initial_cash_quote"])
FIXED_QUOTE = float(REQUEST_EXECUTION["sizing"]["fixed_quote"])
SAFE_PROFIT_PERCENT = float(REQUEST_EXECUTION["profit_lock"]["safe_profit_percent"])
USE_FIXED_QUOTE = REQUEST_EXECUTION["sizing"]["mode"] == "fixed_quote"
USE_PROFIT_LOCK = bool(REQUEST_EXECUTION["profit_lock"]["enabled"])
BARS_PER_YEAR_EXEC_1M = 365.0 * 24.0 * 60.0

DIRECTION_MODE_LONG_ONLY = "long_only"
DIRECTION_MODE_LONG_SHORT_REVERSAL = "long_short_reversal"
DIRECTION_MODES = (DIRECTION_MODE_LONG_ONLY, DIRECTION_MODE_LONG_SHORT_REVERSAL)
DIRECTION_MODE_CODE = {
    DIRECTION_MODE_LONG_ONLY: np.int8(1),
    DIRECTION_MODE_LONG_SHORT_REVERSAL: np.int8(2),
}


def direction_mode_code(direction_mode: str) -> np.int8:
    if direction_mode not in DIRECTION_MODE_CODE:
        raise ValueError(f"Unsupported direction_mode={direction_mode!r}; expected one of {tuple(DIRECTION_MODE_CODE)}.")
    return DIRECTION_MODE_CODE[direction_mode]

CLOSE_ON_END = np.int8(1 if REQUEST_EXECUTION["close_on_end"] else 0)
NEG_INF = np.float32(-1e30)
NEG_LARGE = -1.0e30

RUN_TIME_RANGE_START = REQUEST_PERIOD["start"]
RUN_TIME_RANGE_END = REQUEST_PERIOD["end"]
RUN_TIME_RANGE_START_MS = int(RUN_TIME_RANGE_START.timestamp() * 1000)
RUN_TIME_RANGE_END_MS = int(RUN_TIME_RANGE_END.timestamp() * 1000)


def canonical_json_hash(payload) -> str:
    def normalize(value):
        if isinstance(value, datetime):
            return value.isoformat()
        if isinstance(value, Path):
            return str(value)
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, dict):
            return {str(k): normalize(v) for k, v in sorted(value.items())}
        if isinstance(value, (list, tuple)):
            return [normalize(v) for v in value]
        return value
    encoded = json.dumps(normalize(payload), sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def validate_request_indicators(request_indicators: list[dict], catalog: dict) -> dict:
    specs = {}
    for item in request_indicators:
        indicator_id = item["indicator_id"]
        if indicator_id not in catalog:
            raise KeyError(f"Unknown indicator_id in request: {indicator_id!r}")
        spec = dict(catalog[indicator_id])
        source_set = set(spec["sources"])
        invalid_sources = sorted(set(item["sources"]) - source_set)
        if invalid_sources:
            raise ValueError(f"Invalid sources for {indicator_id!r}: {invalid_sources}")
        start, end = item["window_range"]
        values = spec["param_values"]
        if start > end or start < min(values) or end > max(values):
            raise ValueError(f"Invalid window_range for {indicator_id!r}: {item['window_range']}")
        specs[indicator_id] = spec
    return specs


INDICATOR_SPECS_15M = validate_request_indicators(REQUEST_INDICATORS, INDICATOR_CATALOG_15M)
DEFAULT_EAGER_INDICATOR_IDS_15M = INDICATOR_ARITY_SETS[2]
SIGNAL_PATHS_15M = {
    indicator_id: {
        "manifest": ARTIFACT_ROOT / "signals" / TIMEFRAME / indicator_id / "manifest.yaml",
        "signals": ARTIFACT_ROOT / "signals" / TIMEFRAME / indicator_id / "signals.i8.npy",
    }
    for indicator_id in DEFAULT_EAGER_INDICATOR_IDS_15M
}
REQUEST_HASH = canonical_json_hash(BACKTEST_REQUEST)

PATHS_TO_CHECK = [
    SLOT_MANIFEST_PATH,
    PRICE_OPEN_TIME_15M_PATH,
    PRICE_CLOSE_TIME_15M_PATH,
    PRICE_OHLCV_15M_PATH,
    PRICE_OPEN_TIME_1M_PATH,
    PRICE_CLOSE_TIME_1M_PATH,
    PRICE_OHLCV_1M_PATH,
    BAR_OPEN_1M_IDX_15M_PATH,
    BAR_CLOSE_1M_IDX_15M_PATH,
    HIT_TIMES_MANIFEST_15M_PATH,
    TP_VALUES_15M_PATH,
    SL_VALUES_15M_PATH,
    LONG_TP_15M_PATH,
    LONG_SL_15M_PATH,
    SHORT_TP_15M_PATH,
    SHORT_SL_15M_PATH,
] + [value for item in SIGNAL_PATHS_15M.values() for value in item.values()]

for path in PATHS_TO_CHECK:
    print(f"{path}: exists={path.exists()}")
print("request hash:", REQUEST_HASH)


In [ ]:
slot_manifest = yaml.safe_load(SLOT_MANIFEST_PATH.read_text())
price_manifest_15m = {item["timeframe"]: item for item in slot_manifest["prices"]}[TIMEFRAME]
signal_manifests_15m = {
    indicator_id: yaml.safe_load(paths["manifest"].read_text())
    for indicator_id, paths in SIGNAL_PATHS_15M.items()
}

print("slot:", slot_manifest["slot"], "generation:", slot_manifest["slot_generation"], "asof_date:", slot_manifest["asof_date"])
print("15m price bars:", price_manifest_15m["coverage"]["bar_count"])
for indicator_id, manifest in signal_manifests_15m.items():
    print(indicator_id, "rows_count=", manifest["rows_count"], "shape=", tuple(manifest["signals"]["shape"]))
print("request indicators:", REQUEST_INDICATORS)
print("row prefilter fraction:", PREFILTER_TOP_FRAC)


In [ ]:
price_open_time_15m = np.load(PRICE_OPEN_TIME_15M_PATH, mmap_mode="r" if USE_MMAP else None)
price_close_time_15m = np.load(PRICE_CLOSE_TIME_15M_PATH, mmap_mode="r" if USE_MMAP else None)
price_ohlcv_15m = np.load(PRICE_OHLCV_15M_PATH, mmap_mode="r" if USE_MMAP else None)
price_open_time_1m = np.load(PRICE_OPEN_TIME_1M_PATH, mmap_mode="r" if USE_MMAP else None)
price_close_time_1m = np.load(PRICE_CLOSE_TIME_1M_PATH, mmap_mode="r" if USE_MMAP else None)
price_ohlcv_1m = np.load(PRICE_OHLCV_1M_PATH, mmap_mode="r" if USE_MMAP else None)
bar_open_1m_idx_15m = np.load(BAR_OPEN_1M_IDX_15M_PATH, mmap_mode="r" if USE_MMAP else None)
bar_close_1m_idx_15m = np.load(BAR_CLOSE_1M_IDX_15M_PATH, mmap_mode="r" if USE_MMAP else None)
full_signal_matrices_15m = {
    indicator_id: np.load(paths["signals"], mmap_mode="r" if USE_MMAP else None)
    for indicator_id, paths in SIGNAL_PATHS_15M.items()
}

price_fields_15m = {
    "open": np.asarray(price_ohlcv_15m[:, 0], dtype=np.float32),
    "high": np.asarray(price_ohlcv_15m[:, 1], dtype=np.float32),
    "low": np.asarray(price_ohlcv_15m[:, 2], dtype=np.float32),
    "close": np.asarray(price_ohlcv_15m[:, 3], dtype=np.float32),
    "volume": np.asarray(price_ohlcv_15m[:, 4], dtype=np.float32),
}
price_fields_1m = {
    "open": np.asarray(price_ohlcv_1m[:, 0], dtype=np.float32),
    "close": np.asarray(price_ohlcv_1m[:, 3], dtype=np.float32),
}

time_mask_15m = (price_open_time_15m >= RUN_TIME_RANGE_START_MS) & (price_open_time_15m < RUN_TIME_RANGE_END_MS)
time_mask_idx_15m = np.flatnonzero(time_mask_15m)
if time_mask_idx_15m.size == 0:
    raise ValueError("Run time range selects no 15m bars.")
time_slice_start_15m = int(time_mask_idx_15m[0])
time_slice_stop_15m = int(time_mask_idx_15m[-1]) + 1
USE_CONTIGUOUS_TIME_SLICE_15M = (time_slice_stop_15m - time_slice_start_15m) == int(time_mask_idx_15m.size)
time_selector_15m = slice(time_slice_start_15m, time_slice_stop_15m) if USE_CONTIGUOUS_TIME_SLICE_15M else time_mask_15m
run_bar_open_1m_idx_15m = np.asarray(bar_open_1m_idx_15m[time_selector_15m], dtype=np.int32)
run_bar_close_1m_idx_15m = np.asarray(bar_close_1m_idx_15m[time_selector_15m], dtype=np.int32)

signal_close_15m = np.asarray(price_fields_15m["close"][time_selector_15m], dtype=np.float32)
signal_returns_15m = np.ascontiguousarray(((signal_close_15m[1:] / signal_close_15m[:-1]) - 1.0).astype(np.float32))
n_signal_bars = int(signal_close_15m.shape[0])
n_signal_intervals = int(signal_returns_15m.shape[0])
T_exec_limit_1m = np.int32(int(run_bar_close_1m_idx_15m[-1]) + 1)
last_close_1m = float(price_fields_1m["close"][int(T_exec_limit_1m) - 1])

sig_entry_exec_idx_15m = np.empty(n_signal_bars, dtype=np.int32)
if n_signal_bars > 1:
    sig_entry_exec_idx_15m[:-1] = np.asarray(run_bar_open_1m_idx_15m[1:], dtype=np.int32)
sig_entry_exec_idx_15m[-1] = T_exec_limit_1m

print("signal bars 15m:", n_signal_bars)
print("signal intervals 15m:", n_signal_intervals)
print("execution limit 1m:", int(T_exec_limit_1m))
for indicator_id, matrix in full_signal_matrices_15m.items():
    print(indicator_id, matrix.shape, matrix.dtype)


In [ ]:
def njit_cached(*, parallel: bool = False, fastmath: bool = False, inline: str = "never"):
    """Return a notebook-friendly Numba decorator without filesystem cache dependence.

    Parameters:
        parallel: Whether to enable Numba parallel lowering.
        fastmath: Whether to enable relaxed floating-point optimizations.
        inline: Requested Numba inline policy.

    Returns:
        A decorator wrapping `numba.njit` with `cache=False`.

    Assumptions:
        The notebook executes in an environment where file-backed cache locators may be unavailable.

    Raises:
        None directly; Numba compilation errors propagate from the wrapped function.

    Side effects:
        Triggers Numba compilation on first call of the wrapped function.
    """
    def decorate(func):
        return nb.njit(parallel=parallel, fastmath=fastmath, inline=inline, cache=False)(func)
    return decorate


In [ ]:
def build_row_catalog(*, indicator_id, sources, param_name, param_values):
    """Build row metadata for one artifact-backed indicator matrix.

    Parameters:
        indicator_id: Indicator id matching the artifact directory name.
        sources: Ordered source axis for this artifact matrix.
        param_name: Human-readable parameter axis name.
        param_values: Ordered parameter values inside each source block.

    Returns:
        A list of dictionaries with `row_id`, `source`, and parameter value for every artifact row.

    Assumptions:
        The artifact layout is source-major and parameter-minor inside each source block.

    Raises:
        None.

    Side effects:
        None.
    """
    rows = []
    block = len(param_values)
    for source_idx, source_name in enumerate(sources):
        base = source_idx * block
        for offset, param_value in enumerate(param_values):
            rows.append({
                "indicator_id": indicator_id,
                "row_id": base + offset,
                "source": source_name,
                param_name: int(param_value),
            })
    return rows


def row_ids_for_sources(*, indicator_id, source_names):
    """Return original artifact row ids for the requested source blocks.

    Parameters:
        indicator_id: Indicator id present in `INDICATOR_SPECS_15M`.
        source_names: Iterable of source names to select.

    Returns:
        An `int32` array with the original artifact row ids for the requested sources.

    Assumptions:
        Each source occupies one contiguous block of rows.

    Raises:
        KeyError: If the requested source is not present for the indicator.

    Side effects:
        None.
    """
    meta = INDICATOR_SPECS_15M[indicator_id]
    param_count = len(meta["param_values"])
    source_to_index = {name: idx for idx, name in enumerate(meta["sources"])}
    row_ids = []
    for source_name in source_names:
        source_idx = source_to_index[source_name]
        start = source_idx * param_count
        row_ids.extend(range(start, start + param_count))
    return np.asarray(row_ids, dtype=np.int32)


def extract_signal_rows(*, indicator_id, row_ids):
    """Load a bounded set of 15m signal rows for one indicator into a contiguous matrix.

    Parameters:
        indicator_id: Indicator id present in `full_signal_matrices_15m`.
        row_ids: Original artifact row ids to extract.

    Returns:
        Int8 matrix with shape `(len(row_ids), n_signal_bars)` over the run time range.

    Assumptions:
        `time_selector_15m` already bounds the active request period.

    Raises:
        ValueError: If the row id list is empty.

    Side effects:
        Copies the selected memmap rows into a contiguous in-memory matrix.
    """
    row_ids = np.asarray(row_ids, dtype=np.int32)
    if row_ids.size == 0:
        raise ValueError(f"Empty row selection for {indicator_id!r}.")
    ensure_indicator_loaded(indicator_id)
    signal_matrix = full_signal_matrices_15m[indicator_id]
    if USE_CONTIGUOUS_TIME_SLICE_15M:
        selected = signal_matrix[row_ids, time_slice_start_15m:time_slice_stop_15m]
    else:
        selected = signal_matrix[row_ids][:, time_mask_15m]
    return np.ascontiguousarray(np.asarray(selected, dtype=np.int8))


row_catalogs_15m = {
    indicator_id: build_row_catalog(
        indicator_id=indicator_id,
        sources=meta["sources"],
        param_name=meta["param_name"],
        param_values=meta["param_values"],
    )
    for indicator_id, meta in (
        (indicator_id, INDICATOR_SPECS_15M[indicator_id])
        for indicator_id in DEFAULT_EAGER_INDICATOR_IDS_15M
    )
}

row_counts_15m = {
    indicator_id: int(matrix.shape[0])
    for indicator_id, matrix in full_signal_matrices_15m.items()
}



def signal_paths_for_indicator(indicator_id: str) -> dict:
    """Return manifest and signal paths for a 15m indicator artifact."""
    return {
        "manifest": ARTIFACT_ROOT / "signals" / TIMEFRAME / indicator_id / "manifest.yaml",
        "signals": ARTIFACT_ROOT / "signals" / TIMEFRAME / indicator_id / "signals.i8.npy",
    }


def ensure_indicator_loaded(indicator_id: str) -> None:
    """Lazily load optional indicator metadata, memmap, and row catalog.

    The default notebook opens only a small eager indicator set.
    Wider arity benchmarks call this helper when an extra indicator first appears in
    `row_pools`.
    """
    if indicator_id not in INDICATOR_SPECS_15M:
        raise KeyError(f"Unknown 15m indicator spec: {indicator_id!r}")
    if indicator_id in full_signal_matrices_15m:
        return

    paths = SIGNAL_PATHS_15M.setdefault(indicator_id, signal_paths_for_indicator(indicator_id))
    missing = [name for name, artifact_path in paths.items() if not artifact_path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing artifact files for {indicator_id!r}: {missing}")

    meta = INDICATOR_SPECS_15M[indicator_id]
    signal_manifests_15m[indicator_id] = yaml.safe_load(paths["manifest"].read_text())
    full_signal_matrices_15m[indicator_id] = np.load(paths["signals"], mmap_mode="r" if USE_MMAP else None)
    row_catalogs_15m[indicator_id] = build_row_catalog(
        indicator_id=indicator_id,
        sources=meta["sources"],
        param_name=meta["param_name"],
        param_values=meta["param_values"],
    )
    row_counts_15m[indicator_id] = int(full_signal_matrices_15m[indicator_id].shape[0])

raw_variant_count_eager = math.prod(row_counts_15m.values())
print("eager indicator row counts:", row_counts_15m)
print("plain eager-indicator combinations:", raw_variant_count_eager)
print("sample ma.hma rows:", row_catalogs_15m["ma.hma"][:3])


In [ ]:
def topk_fraction_idx(score: np.ndarray, frac: float) -> np.ndarray:
    """Select top indices by fraction from a one-dimensional score vector.

    Parameters:
        score: One-dimensional score array.
        frac: Fraction in `(0, 1]` describing how many elements to keep.

    Returns:
        Int32 indices of the kept elements.

    Assumptions:
        The input contains at least one finite element.

    Raises:
        ValueError: If `frac` is outside `(0, 1]`.

    Side effects:
        None.
    """
    if not (0.0 < frac <= 1.0):
        raise ValueError(f"frac must be in (0, 1], got {frac!r}")
    n = int(score.shape[0])
    k = max(1, int(math.ceil(n * frac)))
    if k >= n:
        return np.arange(n, dtype=np.int32)
    idx = np.argpartition(score, n - k)[n - k:]
    return np.sort(idx.astype(np.int32))


def single_score_chunked(sig_T_i8: np.ndarray, ret_f32: np.ndarray, chunk: int) -> np.ndarray:
    """Compute a chunked dot-product proxy score for one indicator family.

    Parameters:
        sig_T_i8: Int8 signal matrix shaped `(n_rows, n_intervals)`.
        ret_f32: Float32 return vector shaped `(n_intervals,)`.
        chunk: Chunk length along the time axis.

    Returns:
        Float32 proxy score per row.

    Assumptions:
        The signal matrix is already aligned to the return intervals.

    Raises:
        ValueError: If matrix and return lengths disagree.

    Side effects:
        None.
    """
    if sig_T_i8.shape[1] != ret_f32.shape[0]:
        raise ValueError("Signal matrix and return vector must share the same interval length.")
    n_rows, n_intervals = sig_T_i8.shape
    out = np.zeros(n_rows, dtype=np.float32)
    for t0 in range(0, n_intervals, chunk):
        t1 = min(t0 + chunk, n_intervals)
        out += sig_T_i8[:, t0:t1].astype(np.float32) @ ret_f32[t0:t1]
    return out


@njit_cached(parallel=True, fastmath=False)
def fused_row_prefilter_stats(
    trade_T: np.ndarray,
    ret_15m: np.ndarray,
    out_nonzero: np.ndarray,
    out_proxy: np.ndarray,
    out_change_count: np.ndarray,
) -> None:
    """Compute row activity, proxy score, and change count in one pass."""
    n_rows = trade_T.shape[0]
    n_sig = trade_T.shape[1]
    n_intervals = ret_15m.shape[0]

    for row_idx in nb.prange(n_rows):
        nonzero = np.int32(0)
        proxy = np.float32(0.0)
        change_count = np.int32(0)

        for t in range(n_intervals):
            value = trade_T[row_idx, t]
            if value != 0:
                nonzero += 1
                proxy += np.float32(value) * ret_15m[t]
            next_t = t + 1
            if next_t < n_sig and trade_T[row_idx, next_t] != value:
                change_count += 1

        for t in range(n_intervals + 1, n_sig):
            if trade_T[row_idx, t] != trade_T[row_idx, t - 1]:
                change_count += 1

        out_nonzero[row_idx] = nonzero
        out_proxy[row_idx] = proxy
        out_change_count[row_idx] = change_count


@njit_cached(parallel=True)
def fill_signal_segments_i8(
    trade_T: np.ndarray,
    starts: np.ndarray,
    ends: np.ndarray,
    values: np.ndarray,
    counts: np.ndarray,
) -> None:
    """Fill padded segment arrays for a filtered signal matrix."""
    n_rows = trade_T.shape[0]
    n_sig = trade_T.shape[1]

    for row_idx in nb.prange(n_rows):
        segment_idx = 0
        segment_start = np.int32(0)
        current_value = trade_T[row_idx, 0]
        for t in range(1, n_sig):
            value = trade_T[row_idx, t]
            if value != current_value:
                starts[row_idx, segment_idx] = segment_start
                ends[row_idx, segment_idx] = np.int32(t)
                values[row_idx, segment_idx] = current_value
                segment_idx += 1
                segment_start = np.int32(t)
                current_value = value

        starts[row_idx, segment_idx] = segment_start
        ends[row_idx, segment_idx] = np.int32(n_sig)
        values[row_idx, segment_idx] = current_value
        counts[row_idx] = np.int32(segment_idx + 1)


def build_signal_segments(trade_T: np.ndarray):
    """Build padded change-point segments for an int8 signal matrix."""
    if trade_T.ndim != 2 or trade_T.shape[1] == 0:
        raise ValueError("trade_T must be a non-empty 2D signal matrix.")
    change_count = (trade_T[:, 1:] != trade_T[:, :-1]).sum(axis=1).astype(np.int32)
    counts_expected = change_count + np.int32(1)
    max_segments = int(counts_expected.max())
    starts = np.zeros((trade_T.shape[0], max_segments), dtype=np.int32)
    ends = np.zeros((trade_T.shape[0], max_segments), dtype=np.int32)
    values = np.zeros((trade_T.shape[0], max_segments), dtype=np.int8)
    counts = np.zeros(trade_T.shape[0], dtype=np.int32)
    fill_signal_segments_i8(trade_T, starts, ends, values, counts)
    if not np.array_equal(counts, counts_expected):
        raise AssertionError("Segment count mismatch while compressing signals.")
    return {
        "starts": starts,
        "ends": ends,
        "values": values,
        "counts": counts,
        "change_count": change_count,
    }


def prefilter_indicator_rows(*, trade_T: np.ndarray, indicator_id: str, row_ids: np.ndarray, top_frac: float, min_nonzero: int, fee_rate: float, time_chunk: int):
    """Apply a fused single-indicator prefilter before exact combo evaluation.

    Parameters:
        trade_T: Int8 signal matrix shaped `(n_rows, n_signal_bars)`.
        indicator_id: Indicator id for diagnostics.
        row_ids: Original artifact row ids aligned to `trade_T` rows.
        top_frac: Fraction of rows to retain after scoring.
        min_nonzero: Minimum number of non-zero signals required to keep a row.
        fee_rate: Per-side fee used as a rough penalty term in the proxy score.
        time_chunk: Kept for call-site compatibility; fused stats scan the row once.

    Returns:
        Standard indicator-pool dictionary without heavy derived artifacts attached yet.

    Assumptions:
        The exact path trades on the next bar, so proxy evaluation uses `trade_T[:, :-1]`.

    Raises:
        ValueError: If no candidate survives the non-zero filter.

    Side effects:
        None.
    """
    row_ids = np.asarray(row_ids, dtype=np.int32)
    if trade_T.shape[0] != row_ids.shape[0]:
        raise ValueError(f"Row id alignment mismatch for {indicator_id!r}.")

    nonzero = np.empty(trade_T.shape[0], dtype=np.int32)
    proxy = np.empty(trade_T.shape[0], dtype=np.float32)
    change_count = np.empty(trade_T.shape[0], dtype=np.int32)
    fused_row_prefilter_stats(trade_T, signal_returns_15m, nonzero, proxy, change_count)

    adjusted = proxy - (fee_rate * nonzero.astype(np.float32))
    valid = nonzero >= int(min_nonzero)
    if not np.any(valid):
        raise ValueError(f"No rows survive min_nonzero={min_nonzero} for {indicator_id!r}.")

    valid_idx = np.flatnonzero(valid)
    keep_from_valid = topk_fraction_idx(adjusted[valid_idx], top_frac)
    keep_idx = np.sort(valid_idx[keep_from_valid].astype(np.int32))

    filtered_row_ids = np.ascontiguousarray(row_ids[keep_idx])
    filtered_trade_T = np.ascontiguousarray(trade_T[keep_idx])
    filtered_eval_T = np.ascontiguousarray(filtered_trade_T[:, :n_signal_intervals])
    row_score = np.ascontiguousarray(adjusted[keep_idx])
    filtered_nonzero = np.ascontiguousarray(nonzero[keep_idx])
    filtered_change_count = np.ascontiguousarray(change_count[keep_idx])

    return {
        "indicator_id": indicator_id,
        "row_ids": filtered_row_ids,
        "filtered_row_ids": filtered_row_ids,
        "trade_T": filtered_trade_T,
        "eval_T": filtered_eval_T,
        "segments": None,
        "row_score": row_score,
        "score_adj": row_score,
        "nonzero": filtered_nonzero,
        "change_count": filtered_change_count,
        "metadata": None,
    }


INDICATOR_POOL_SCHEMA_KEYS = (
    "indicator_id",
    "row_ids",
    "trade_T",
    "eval_T",
    "segments",
    "row_score",
    "nonzero",
    "change_count",
    "metadata",
)


def attach_indicator_pool_artifacts(pool: dict) -> dict:
    """Attach universal per-row metadata and compressed signal segments to an indicator pool."""
    indicator_id = pool["indicator_id"]
    segments = build_signal_segments(pool["trade_T"])
    if not np.array_equal(segments["change_count"], pool["change_count"]):
        raise AssertionError(f"Segment change counts differ from fused row stats for {indicator_id!r}.")
    pool["segments"] = segments
    pool["metadata"] = [
        row_catalogs_15m[indicator_id][int(row_id)]
        for row_id in pool["row_ids"]
    ]
    validate_indicator_pool_schema(pool)
    return pool


def validate_indicator_pool_schema(pool: dict) -> None:
    """Validate the standard indicator-pool shape used by search and exact backends."""
    missing = [key for key in INDICATOR_POOL_SCHEMA_KEYS if key not in pool]
    if missing:
        raise KeyError(f"Indicator pool {pool.get('indicator_id')!r} is missing keys: {missing}")

    n_rows = int(pool["trade_T"].shape[0])
    if pool["row_ids"].shape[0] != n_rows:
        raise ValueError(f"row_ids length mismatch for {pool['indicator_id']!r}.")
    if pool["eval_T"].shape[0] != n_rows or pool["eval_T"].shape[1] != n_signal_intervals:
        raise ValueError(f"eval_T shape mismatch for {pool['indicator_id']!r}.")
    if pool["row_score"].shape[0] != n_rows:
        raise ValueError(f"row_score length mismatch for {pool['indicator_id']!r}.")
    if pool["nonzero"].shape[0] != n_rows:
        raise ValueError(f"nonzero length mismatch for {pool['indicator_id']!r}.")
    if pool["change_count"].shape[0] != n_rows:
        raise ValueError(f"change_count length mismatch for {pool['indicator_id']!r}.")
    if len(pool["metadata"]) != n_rows:
        raise ValueError(f"metadata length mismatch for {pool['indicator_id']!r}.")

    segments = pool["segments"]
    for key in ("starts", "ends", "values", "counts", "change_count"):
        if key not in segments:
            raise KeyError(f"segments for {pool['indicator_id']!r} are missing {key!r}.")
    if segments["starts"].shape[0] != n_rows or segments["ends"].shape[0] != n_rows or segments["values"].shape[0] != n_rows:
        raise ValueError(f"segment matrix row mismatch for {pool['indicator_id']!r}.")
    if segments["counts"].shape[0] != n_rows:
        raise ValueError(f"segment counts length mismatch for {pool['indicator_id']!r}.")


def prepare_indicator_pool(*, indicator_id: str, row_ids: np.ndarray | None = None, top_frac: float = PREFILTER_TOP_FRAC, min_nonzero: int = PREFILTER_MIN_NONZERO, fee_rate: float = FEE_RATE, time_chunk: int = TIME_CHUNK):
    """Load one indicator into the standard pool format used by search backends.

    Parameters:
        indicator_id: Indicator id to prepare.
        row_ids: Optional original artifact row ids. When omitted, all rows are used.
        top_frac: Fraction of rows to keep after proxy scoring.
        min_nonzero: Minimum count of non-zero signals required for a row.
        fee_rate: Fee penalty used inside the proxy prefilter.
        time_chunk: Kept for compatibility with older chunked proxy scoring.

    Returns:
        Standard indicator-pool dictionary with row ids, matrices, segments, stats, and metadata.

    Assumptions:
        Row metadata is available in `row_catalogs_15m`.

    Raises:
        ValueError: If the initial row selection is empty.

    Side effects:
        Loads the selected signal rows into memory and builds compressed signal segments.
    """
    if row_ids is None:
        row_ids = np.arange(row_counts_15m[indicator_id], dtype=np.int32)
    row_ids = np.asarray(row_ids, dtype=np.int32)
    if row_ids.size == 0:
        raise ValueError(f"Empty requested pool for {indicator_id!r}.")
    trade_T = extract_signal_rows(indicator_id=indicator_id, row_ids=row_ids)
    pool = prefilter_indicator_rows(
        trade_T=trade_T,
        indicator_id=indicator_id,
        row_ids=row_ids,
        top_frac=top_frac,
        min_nonzero=min_nonzero,
        fee_rate=fee_rate,
        time_chunk=time_chunk,
    )
    return attach_indicator_pool_artifacts(pool)


def prepare_indicator_pools(*, indicator_ids: tuple[str, ...], row_pools: dict, top_frac: float, min_nonzero: int, fee_rate: float, time_chunk: int) -> dict:
    """Prepare multiple indicators into the same standard pool schema."""
    indicator_pools = {}
    for indicator_id in indicator_ids:
        requested_rows = np.asarray(row_pools[indicator_id], dtype=np.int32)
        indicator_pools[indicator_id] = prepare_indicator_pool(
            indicator_id=indicator_id,
            row_ids=requested_rows,
            top_frac=top_frac,
            min_nonzero=min_nonzero,
            fee_rate=fee_rate,
            time_chunk=time_chunk,
        )
    return indicator_pools


In [ ]:
@njit_cached(inline="always")
def consensus_dir2(dema_value: np.int8, hma_value: np.int8) -> np.int8:
    """Resolve the two-indicator consensus direction for one signal bar.

    Parameters:
        dema_value: DEMA signal on one bar.
        hma_value: HMA signal on one bar.

    Returns:
        `1` for unanimous long, `-1` for unanimous short, `0` otherwise.

    Assumptions:
        Inputs are already normalized to `{-1, 0, 1}`.

    Raises:
        None.

    Side effects:
        None.
    """
    if dema_value == 1 and hma_value == 1:
        return np.int8(1)
    if dema_value == -1 and hma_value == -1:
        return np.int8(-1)
    return np.int8(0)

@njit_cached(inline="always")
def apply_direction_mode(raw_dir: np.int8, direction_mode: np.int8) -> np.int8:
    """Map raw consensus direction into the requested trading direction mode."""
    if direction_mode == 1:
        if raw_dir == 1:
            return np.int8(1)
        return np.int8(0)
    return raw_dir


@njit_cached()
def build_trade_list_for_two_rows(
    dema_sig_row: np.ndarray,
    hma_sig_row: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    T_exec: np.int32,
    out_entry_exec_idx: np.ndarray,
    out_dir: np.ndarray,
    out_sig_exit_exec_idx: np.ndarray,
) -> np.int32:
    """Build a compact trade list for one two-indicator consensus strategy.

    Parameters:
        dema_sig_row: DEMA signal row over the active 15m run range.
        hma_sig_row: HMA signal row over the active 15m run range.
        sig_entry_exec_idx: Mapping from each 15m signal bar to the next 1m execution entry index.
        T_exec: Exclusive 1m execution limit for the run range.
        out_entry_exec_idx: Preallocated output for trade entry indices.
        out_dir: Preallocated output for trade directions.
        out_sig_exit_exec_idx: Preallocated output for signal-exit indices.

    Returns:
        Number of trades written into the output buffers, or `-1` when capacity is insufficient.

    Assumptions:
        Repeated confirmations in the same direction while already in a position are ignored.

    Raises:
        None directly; buffer overflow is reported via `-1`.

    Side effects:
        Mutates the provided output arrays in place.
    """
    n_sig = dema_sig_row.shape[0]
    n_trades = np.int32(0)
    current_dir = np.int8(0)
    current_entry = np.int32(0)

    for t in range(n_sig):
        dirn = consensus_dir2(dema_sig_row[t], hma_sig_row[t])
        if dirn == 0:
            continue

        entry_exec = sig_entry_exec_idx[t]
        if entry_exec >= T_exec:
            break

        if current_dir == 0:
            current_dir = dirn
            current_entry = np.int32(entry_exec)
            continue

        if dirn == current_dir:
            continue

        if n_trades >= out_entry_exec_idx.shape[0]:
            return np.int32(-1)

        out_entry_exec_idx[n_trades] = current_entry
        out_dir[n_trades] = current_dir
        out_sig_exit_exec_idx[n_trades] = np.int32(entry_exec)
        n_trades += 1
        current_dir = dirn
        current_entry = np.int32(entry_exec)

    if current_dir != 0:
        if n_trades >= out_entry_exec_idx.shape[0]:
            return np.int32(-1)
        out_entry_exec_idx[n_trades] = current_entry
        out_dir[n_trades] = current_dir
        out_sig_exit_exec_idx[n_trades] = T_exec
        n_trades += 1

    return n_trades


@njit_cached(parallel=True)
def count_trades_for_two_combos(
    combo_dema_idx: np.ndarray,
    combo_hma_idx: np.ndarray,
    dema_trade_T: np.ndarray,
    hma_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    T_exec: np.int32,
    out_trade_counts: np.ndarray,
) -> None:
    """Count compressed trades for a chunk of two-indicator combinations.

    Parameters:
        combo_dema_idx: Row indices into the filtered DEMA trade matrix.
        combo_hma_idx: Row indices into the filtered HMA trade matrix.
        dema_trade_T: Filtered DEMA trade matrix shaped `(n_rows, n_signal_bars)`.
        hma_trade_T: Filtered HMA trade matrix shaped `(n_rows, n_signal_bars)`.
        sig_entry_exec_idx: 15m signal-bar to 1m execution-entry mapping.
        T_exec: Exclusive 1m execution limit for the run range.
        out_trade_counts: Output vector for trade counts per combination.

    Returns:
        None.

    Assumptions:
        Both combo index arrays share the same length.

    Raises:
        None.

    Side effects:
        Writes counts into `out_trade_counts`.
    """
    K = combo_dema_idx.shape[0]
    n_sig = dema_trade_T.shape[1]

    for k in nb.prange(K):
        current_dir = np.int8(0)
        n_trades = np.int32(0)
        di = combo_dema_idx[k]
        hi = combo_hma_idx[k]

        for t in range(n_sig):
            dirn = consensus_dir2(dema_trade_T[di, t], hma_trade_T[hi, t])
            if dirn == 0:
                continue
            if sig_entry_exec_idx[t] >= T_exec:
                break
            if current_dir == 0:
                current_dir = dirn
                continue
            if dirn != current_dir:
                n_trades += 1
                current_dir = dirn

        if current_dir != 0:
            n_trades += 1
        out_trade_counts[k] = n_trades


In [ ]:
@njit_cached(inline="always")
def trade_sharpe_kernel(trade_count: np.int32, sum_trade_return: float, sum_trade_return_squared: float, bars_per_year_exec: float, sentinel_index: np.int32) -> float:
    """Compute trade-level Sharpe from accumulated returns.

    Parameters:
        trade_count: Number of closed trades.
        sum_trade_return: Sum of per-trade decimal returns.
        sum_trade_return_squared: Sum of squared per-trade decimal returns.
        bars_per_year_exec: Annualization denominator in execution bars.
        sentinel_index: Total execution bars in the replay window.

    Returns:
        Trade-level Sharpe ratio.

    Assumptions:
        The caller already filtered invalid or empty trade sets.

    Raises:
        None.

    Side effects:
        None.
    """
    if trade_count <= 1:
        return 0.0
    mean_trade_return = sum_trade_return / float(trade_count)
    variance = (sum_trade_return_squared / float(trade_count)) - (mean_trade_return * mean_trade_return)
    if variance <= 0.0:
        return 0.0
    years = float(sentinel_index) / float(bars_per_year_exec)
    if years <= 0.0:
        years = 1.0
    trades_per_year = float(trade_count) / years
    return (mean_trade_return / math.sqrt(variance)) * math.sqrt(trades_per_year)


@njit_cached(inline="always")
def score_trade_list_no_risk(
    entry_exec_idx: np.ndarray,
    dir_arr: np.ndarray,
    sig_exit_exec_idx: np.ndarray,
    n_trades: np.int32,
    exec_open_1m: np.ndarray,
    exec_close_1m: np.ndarray,
    T_exec: np.int32,
    init_cash_quote: float,
    fixed_quote: float,
    fee_rate: float,
    slippage_rate: float,
    safe_profit_percent: float,
    use_fixed_quote: np.int8,
    use_profit_lock: np.int8,
    bars_per_year_exec: float,
    close_on_end: np.int8,
) -> tuple[float, float, float, float, np.int32, float, float, float, float, float]:
    """Score one compact trade list with no-risk execution semantics.

    Parameters:
        entry_exec_idx: Entry execution indexes for one combo.
        dir_arr: Trade directions for one combo.
        sig_exit_exec_idx: Signal-exit execution indexes for one combo.
        n_trades: Number of valid trades.
        exec_open_1m: Full 1m open series.
        exec_close_1m: Full 1m close series.
        T_exec: Exclusive 1m execution limit for the run range.
        init_cash_quote: Initial quote balance.
        fixed_quote: Fixed quote size when fixed sizing is enabled.
        fee_rate: Decimal per-side fee rate.
        slippage_rate: Decimal slippage rate.
        safe_profit_percent: Profit-lock percentage.
        use_fixed_quote: Whether fixed quote sizing is enabled.
        use_profit_lock: Whether profit lock is enabled.
        bars_per_year_exec: Annualization denominator in execution bars.
        close_on_end: Whether a final open trade closes at the last execution close.

    Returns:
        Tuple of no-risk metrics aligned with the persisted summary metrics contract.

    Assumptions:
        Long entries buy above the open by slippage, short entries sell below the open by slippage.

    Raises:
        None.

    Side effects:
        None.
    """
    available_quote = init_cash_quote
    safe_quote = 0.0
    equity = init_cash_quote
    peak_equity = equity
    max_drawdown_pct = 0.0
    gross_profit_quote = 0.0
    gross_loss_quote = 0.0
    closed_trade_count = np.int32(0)
    win_count = np.int32(0)
    sum_trade_return = 0.0
    sum_trade_return_squared = 0.0
    total_trade_return_pct = 0.0
    total_trade_exec_bars = 0.0
    exposure_bars = 0.0

    for trade_index in range(n_trades):
        entry_idx = np.int32(entry_exec_idx[trade_index])
        if entry_idx >= T_exec:
            continue
        exit_idx = np.int32(sig_exit_exec_idx[trade_index])
        if exit_idx < T_exec:
            exit_exec_idx = exit_idx
            exit_price_raw = float(exec_open_1m[exit_exec_idx])
        elif close_on_end == 1 and T_exec > 0:
            exit_exec_idx = np.int32(T_exec - 1)
            exit_price_raw = float(exec_close_1m[exit_exec_idx])
        else:
            continue

        if available_quote <= 0.0:
            continue
        quote_amount = available_quote
        if use_fixed_quote == 1 and fixed_quote < quote_amount:
            quote_amount = fixed_quote
        if quote_amount <= 0.0:
            continue

        trade_direction = np.int8(dir_arr[trade_index])
        entry_price_raw = float(exec_open_1m[entry_idx])
        if trade_direction == 1:
            entry_fill_price = entry_price_raw * (1.0 + slippage_rate)
            exit_fill_price = exit_price_raw * (1.0 - slippage_rate)
        else:
            entry_fill_price = entry_price_raw * (1.0 - slippage_rate)
            exit_fill_price = exit_price_raw * (1.0 + slippage_rate)

        qty_base = quote_amount / entry_fill_price
        entry_fee_quote = quote_amount * fee_rate
        available_quote -= quote_amount + entry_fee_quote

        exit_quote_amount = qty_base * exit_fill_price
        exit_fee_quote = exit_quote_amount * fee_rate
        if trade_direction == 1:
            gross_pnl_quote = exit_quote_amount - quote_amount
        else:
            gross_pnl_quote = quote_amount - exit_quote_amount
        available_quote += quote_amount + gross_pnl_quote - exit_fee_quote
        net_pnl_quote = gross_pnl_quote - entry_fee_quote - exit_fee_quote

        if use_profit_lock == 1 and net_pnl_quote > 0.0:
            locked_profit_quote = net_pnl_quote * (safe_profit_percent / 100.0)
            available_quote -= locked_profit_quote
            safe_quote += locked_profit_quote

        equity = available_quote + safe_quote
        if equity > peak_equity:
            peak_equity = equity
        elif peak_equity > 0.0:
            drawdown_pct = ((peak_equity - equity) / peak_equity) * 100.0
            if drawdown_pct > max_drawdown_pct:
                max_drawdown_pct = drawdown_pct

        trade_return_pct = (net_pnl_quote / quote_amount) * 100.0
        trade_return = net_pnl_quote / quote_amount
        bars_held = float(exit_exec_idx - entry_idx)
        if bars_held < 0.0:
            bars_held = 0.0

        closed_trade_count += 1
        if net_pnl_quote > 0.0:
            win_count += 1
            gross_profit_quote += net_pnl_quote
        elif net_pnl_quote < 0.0:
            gross_loss_quote += abs(net_pnl_quote)
        sum_trade_return += trade_return
        sum_trade_return_squared += trade_return * trade_return
        total_trade_return_pct += trade_return_pct
        total_trade_exec_bars += bars_held
        exposure_bars += bars_held

    total_return_pct = ((equity / init_cash_quote) - 1.0) * 100.0
    if gross_loss_quote > 0.0:
        profit_factor = gross_profit_quote / gross_loss_quote
    elif gross_profit_quote > 0.0:
        profit_factor = np.inf
    else:
        profit_factor = 0.0

    if max_drawdown_pct > 0.0:
        return_over_max_drawdown = total_return_pct / max_drawdown_pct
    elif total_return_pct > 0.0:
        return_over_max_drawdown = np.inf
    else:
        return_over_max_drawdown = 0.0

    if closed_trade_count > 0:
        win_rate_pct = (float(win_count) / float(closed_trade_count)) * 100.0
        avg_trade_ret_pct = total_trade_return_pct / float(closed_trade_count)
        avg_trade_exec_bars = total_trade_exec_bars / float(closed_trade_count)
    else:
        win_rate_pct = 0.0
        avg_trade_ret_pct = 0.0
        avg_trade_exec_bars = 0.0

    exposure_pct = (exposure_bars / float(T_exec)) * 100.0 if T_exec > 0 else 0.0
    sharpe_trades = trade_sharpe_kernel(closed_trade_count, sum_trade_return, sum_trade_return_squared, bars_per_year_exec, T_exec)
    return (
        total_return_pct,
        max_drawdown_pct,
        return_over_max_drawdown,
        profit_factor,
        closed_trade_count,
        sharpe_trades,
        win_rate_pct,
        avg_trade_ret_pct,
        avg_trade_exec_bars,
        exposure_pct,
    )


@njit_cached(parallel=True, fastmath=True)
def evaluate_no_risk_trade_list_fast_two(
    combo_dema_idx: np.ndarray,
    combo_hma_idx: np.ndarray,
    dema_trade_T: np.ndarray,
    hma_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open_1m: np.ndarray,
    exec_close_1m: np.ndarray,
    T_exec: np.int32,
    trade_counts: np.ndarray,
    init_cash_quote: float,
    fixed_quote: float,
    fee_rate: float,
    slippage_rate: float,
    safe_profit_percent: float,
    use_fixed_quote: np.int8,
    use_profit_lock: np.int8,
    bars_per_year_exec: float,
    close_on_end: np.int8,
    out_total_return_pct: np.ndarray,
    out_max_drawdown_pct: np.ndarray,
    out_return_over_max_drawdown: np.ndarray,
    out_profit_factor: np.ndarray,
    out_trade_count: np.ndarray,
    out_sharpe_trades: np.ndarray,
    out_win_rate_pct: np.ndarray,
    out_avg_trade_ret_pct: np.ndarray,
    out_avg_trade_exec_bars: np.ndarray,
    out_exposure_pct: np.ndarray,
) -> None:
    """Score a chunk of two-indicator combos with no-risk execution metrics.

    Parameters:
        combo_dema_idx: Local DEMA row indices.
        combo_hma_idx: Local HMA row indices.
        dema_trade_T: Filtered DEMA trade matrix.
        hma_trade_T: Filtered HMA trade matrix.
        sig_entry_exec_idx: 15m signal-bar to 1m execution-entry mapping.
        exec_open_1m: Full 1m open series.
        exec_close_1m: Full 1m close series.
        T_exec: Exclusive 1m execution limit for the run range.
        trade_counts: Precomputed compact trade counts per combo.
        init_cash_quote: Initial quote balance.
        fixed_quote: Fixed quote size when fixed sizing is enabled.
        fee_rate: Decimal per-side fee rate.
        slippage_rate: Decimal slippage rate.
        safe_profit_percent: Profit-lock percentage.
        use_fixed_quote: Whether fixed quote sizing is enabled.
        use_profit_lock: Whether profit lock is enabled.
        bars_per_year_exec: Annualization denominator in execution bars.
        close_on_end: Whether a final open trade closes at the last execution close.
        out_*: Output arrays for the scored metrics.

    Returns:
        None.

    Assumptions:
        Each combo is independent and can be scored in parallel.

    Raises:
        None.

    Side effects:
        Writes metrics into the output arrays.
    """
    K = combo_dema_idx.shape[0]
    n_sig = dema_trade_T.shape[1]

    for k in nb.prange(K):
        alloc_n = np.int32(trade_counts[k])
        if alloc_n <= 0:
            out_total_return_pct[k] = 0.0
            out_max_drawdown_pct[k] = 0.0
            out_return_over_max_drawdown[k] = 0.0
            out_profit_factor[k] = 0.0
            out_trade_count[k] = 0
            out_sharpe_trades[k] = 0.0
            out_win_rate_pct[k] = 0.0
            out_avg_trade_ret_pct[k] = 0.0
            out_avg_trade_exec_bars[k] = 0.0
            out_exposure_pct[k] = 0.0
            continue

        di = combo_dema_idx[k]
        hi = combo_hma_idx[k]
        entry_arr = np.empty(alloc_n, dtype=np.int32)
        dir_arr = np.empty(alloc_n, dtype=np.int8)
        sig_exit_arr = np.empty(alloc_n, dtype=np.int32)
        n_trades = build_trade_list_for_two_rows(
            dema_trade_T[di],
            hma_trade_T[hi],
            sig_entry_exec_idx,
            T_exec,
            entry_arr,
            dir_arr,
            sig_exit_arr,
        )
        if n_trades <= 0:
            out_total_return_pct[k] = 0.0
            out_max_drawdown_pct[k] = 0.0
            out_return_over_max_drawdown[k] = 0.0
            out_profit_factor[k] = 0.0
            out_trade_count[k] = 0
            out_sharpe_trades[k] = 0.0
            out_win_rate_pct[k] = 0.0
            out_avg_trade_ret_pct[k] = 0.0
            out_avg_trade_exec_bars[k] = 0.0
            out_exposure_pct[k] = 0.0
            continue

        metrics = score_trade_list_no_risk(
            entry_arr,
            dir_arr,
            sig_exit_arr,
            n_trades,
            exec_open_1m,
            exec_close_1m,
            T_exec,
            init_cash_quote,
            fixed_quote,
            fee_rate,
            slippage_rate,
            safe_profit_percent,
            use_fixed_quote,
            use_profit_lock,
            bars_per_year_exec,
            close_on_end,
        )
        out_total_return_pct[k] = metrics[0]
        out_max_drawdown_pct[k] = metrics[1]
        out_return_over_max_drawdown[k] = metrics[2]
        out_profit_factor[k] = metrics[3]
        out_trade_count[k] = metrics[4]
        out_sharpe_trades[k] = metrics[5]
        out_win_rate_pct[k] = metrics[6]
        out_avg_trade_ret_pct[k] = metrics[7]
        out_avg_trade_exec_bars[k] = metrics[8]
        out_exposure_pct[k] = metrics[9]



@njit_cached(inline="always")
def apply_no_risk_trade_to_state(
    entry_idx: np.int32,
    trade_direction: np.int8,
    exit_exec_idx: np.int32,
    exit_price_raw: float,
    exec_open_1m: np.ndarray,
    available_quote: float,
    safe_quote: float,
    equity: float,
    peak_equity: float,
    max_drawdown_pct: float,
    gross_profit_quote: float,
    gross_loss_quote: float,
    closed_trade_count: np.int32,
    win_count: np.int32,
    sum_trade_return: float,
    sum_trade_return_squared: float,
    total_trade_return_pct: float,
    total_trade_exec_bars: float,
    exposure_bars: float,
    init_cash_quote: float,
    fixed_quote: float,
    fee_rate: float,
    slippage_rate: float,
    safe_profit_percent: float,
    use_fixed_quote: np.int8,
    use_profit_lock: np.int8,
) -> tuple[float, float, float, float, float, float, float, np.int32, np.int32, float, float, float, float, float]:
    """Apply one closed no-risk trade directly to aggregate metric state."""
    if available_quote <= 0.0:
        return (
            available_quote,
            safe_quote,
            equity,
            peak_equity,
            max_drawdown_pct,
            gross_profit_quote,
            gross_loss_quote,
            closed_trade_count,
            win_count,
            sum_trade_return,
            sum_trade_return_squared,
            total_trade_return_pct,
            total_trade_exec_bars,
            exposure_bars,
        )

    quote_amount = available_quote
    if use_fixed_quote == 1 and fixed_quote < quote_amount:
        quote_amount = fixed_quote
    if quote_amount <= 0.0:
        return (
            available_quote,
            safe_quote,
            equity,
            peak_equity,
            max_drawdown_pct,
            gross_profit_quote,
            gross_loss_quote,
            closed_trade_count,
            win_count,
            sum_trade_return,
            sum_trade_return_squared,
            total_trade_return_pct,
            total_trade_exec_bars,
            exposure_bars,
        )

    entry_price_raw = float(exec_open_1m[entry_idx])
    if trade_direction == 1:
        entry_fill_price = entry_price_raw * (1.0 + slippage_rate)
        exit_fill_price = exit_price_raw * (1.0 - slippage_rate)
    else:
        entry_fill_price = entry_price_raw * (1.0 - slippage_rate)
        exit_fill_price = exit_price_raw * (1.0 + slippage_rate)

    qty_base = quote_amount / entry_fill_price
    entry_fee_quote = quote_amount * fee_rate
    available_quote -= quote_amount + entry_fee_quote

    exit_quote_amount = qty_base * exit_fill_price
    exit_fee_quote = exit_quote_amount * fee_rate
    if trade_direction == 1:
        gross_pnl_quote = exit_quote_amount - quote_amount
    else:
        gross_pnl_quote = quote_amount - exit_quote_amount
    available_quote += quote_amount + gross_pnl_quote - exit_fee_quote
    net_pnl_quote = gross_pnl_quote - entry_fee_quote - exit_fee_quote

    if use_profit_lock == 1 and net_pnl_quote > 0.0:
        locked_profit_quote = net_pnl_quote * (safe_profit_percent / 100.0)
        available_quote -= locked_profit_quote
        safe_quote += locked_profit_quote

    equity = available_quote + safe_quote
    if equity > peak_equity:
        peak_equity = equity
    elif peak_equity > 0.0:
        drawdown_pct = ((peak_equity - equity) / peak_equity) * 100.0
        if drawdown_pct > max_drawdown_pct:
            max_drawdown_pct = drawdown_pct

    trade_return_pct = (net_pnl_quote / quote_amount) * 100.0
    trade_return = net_pnl_quote / quote_amount
    bars_held = float(exit_exec_idx - entry_idx)
    if bars_held < 0.0:
        bars_held = 0.0

    closed_trade_count += 1
    if net_pnl_quote > 0.0:
        win_count += 1
        gross_profit_quote += net_pnl_quote
    elif net_pnl_quote < 0.0:
        gross_loss_quote += abs(net_pnl_quote)
    sum_trade_return += trade_return
    sum_trade_return_squared += trade_return * trade_return
    total_trade_return_pct += trade_return_pct
    total_trade_exec_bars += bars_held
    exposure_bars += bars_held

    return (
        available_quote,
        safe_quote,
        equity,
        peak_equity,
        max_drawdown_pct,
        gross_profit_quote,
        gross_loss_quote,
        closed_trade_count,
        win_count,
        sum_trade_return,
        sum_trade_return_squared,
        total_trade_return_pct,
        total_trade_exec_bars,
        exposure_bars,
    )


@njit_cached(parallel=True, fastmath=True)
def evaluate_no_risk_streaming_two(
    combo_dema_idx: np.ndarray,
    combo_hma_idx: np.ndarray,
    dema_trade_T: np.ndarray,
    hma_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open_1m: np.ndarray,
    exec_close_1m: np.ndarray,
    T_exec: np.int32,
    init_cash_quote: float,
    fixed_quote: float,
    fee_rate: float,
    slippage_rate: float,
    safe_profit_percent: float,
    use_fixed_quote: np.int8,
    use_profit_lock: np.int8,
    bars_per_year_exec: float,
    close_on_end: np.int8,
    out_total_return_pct: np.ndarray,
    out_max_drawdown_pct: np.ndarray,
    out_return_over_max_drawdown: np.ndarray,
    out_profit_factor: np.ndarray,
    out_trade_count: np.ndarray,
    out_sharpe_trades: np.ndarray,
    out_win_rate_pct: np.ndarray,
    out_avg_trade_ret_pct: np.ndarray,
    out_avg_trade_exec_bars: np.ndarray,
    out_exposure_pct: np.ndarray,
) -> None:
    """Score two-indicator combos in one pass without a separate trade-count pass or trade-list arrays."""
    K = combo_dema_idx.shape[0]
    n_sig = dema_trade_T.shape[1]

    for k in nb.prange(K):
        di = combo_dema_idx[k]
        hi = combo_hma_idx[k]
        available_quote = init_cash_quote
        safe_quote = 0.0
        equity = init_cash_quote
        peak_equity = equity
        max_drawdown_pct = 0.0
        gross_profit_quote = 0.0
        gross_loss_quote = 0.0
        closed_trade_count = np.int32(0)
        win_count = np.int32(0)
        sum_trade_return = 0.0
        sum_trade_return_squared = 0.0
        total_trade_return_pct = 0.0
        total_trade_exec_bars = 0.0
        exposure_bars = 0.0
        current_dir = np.int8(0)
        current_entry = np.int32(0)

        for t in range(n_sig):
            dirn = consensus_dir2(dema_trade_T[di, t], hma_trade_T[hi, t])
            if dirn == 0:
                continue
            entry_exec = sig_entry_exec_idx[t]
            if entry_exec >= T_exec:
                break
            if current_dir == 0:
                current_dir = dirn
                current_entry = np.int32(entry_exec)
                continue
            if dirn == current_dir:
                continue

            (
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
            ) = apply_no_risk_trade_to_state(
                current_entry,
                current_dir,
                np.int32(entry_exec),
                float(exec_open_1m[entry_exec]),
                exec_open_1m,
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
                init_cash_quote,
                fixed_quote,
                fee_rate,
                slippage_rate,
                safe_profit_percent,
                use_fixed_quote,
                use_profit_lock,
            )
            current_dir = dirn
            current_entry = np.int32(entry_exec)

        if current_dir != 0 and close_on_end == 1 and T_exec > 0:
            exit_exec_idx = np.int32(T_exec - 1)
            (
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
            ) = apply_no_risk_trade_to_state(
                current_entry,
                current_dir,
                exit_exec_idx,
                float(exec_close_1m[exit_exec_idx]),
                exec_open_1m,
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
                init_cash_quote,
                fixed_quote,
                fee_rate,
                slippage_rate,
                safe_profit_percent,
                use_fixed_quote,
                use_profit_lock,
            )

        total_return_pct = ((equity / init_cash_quote) - 1.0) * 100.0
        if gross_loss_quote > 0.0:
            profit_factor = gross_profit_quote / gross_loss_quote
        elif gross_profit_quote > 0.0:
            profit_factor = np.inf
        else:
            profit_factor = 0.0

        if max_drawdown_pct > 0.0:
            return_over_max_drawdown = total_return_pct / max_drawdown_pct
        elif total_return_pct > 0.0:
            return_over_max_drawdown = np.inf
        else:
            return_over_max_drawdown = 0.0

        if closed_trade_count > 0:
            win_rate_pct = (float(win_count) / float(closed_trade_count)) * 100.0
            avg_trade_ret_pct = total_trade_return_pct / float(closed_trade_count)
            avg_trade_exec_bars = total_trade_exec_bars / float(closed_trade_count)
        else:
            win_rate_pct = 0.0
            avg_trade_ret_pct = 0.0
            avg_trade_exec_bars = 0.0

        exposure_pct = (exposure_bars / float(T_exec)) * 100.0 if T_exec > 0 else 0.0
        sharpe_trades = trade_sharpe_kernel(
            closed_trade_count,
            sum_trade_return,
            sum_trade_return_squared,
            bars_per_year_exec,
            T_exec,
        )
        out_total_return_pct[k] = total_return_pct
        out_max_drawdown_pct[k] = max_drawdown_pct
        out_return_over_max_drawdown[k] = return_over_max_drawdown
        out_profit_factor[k] = profit_factor
        out_trade_count[k] = closed_trade_count
        out_sharpe_trades[k] = sharpe_trades
        out_win_rate_pct[k] = win_rate_pct
        out_avg_trade_ret_pct[k] = avg_trade_ret_pct
        out_avg_trade_exec_bars[k] = avg_trade_exec_bars
        out_exposure_pct[k] = exposure_pct




@njit_cached(parallel=True, fastmath=True)
def evaluate_no_risk_event_segments_two(
    combo_dema_idx: np.ndarray,
    combo_hma_idx: np.ndarray,
    dema_segment_starts: np.ndarray,
    dema_segment_ends: np.ndarray,
    dema_segment_values: np.ndarray,
    dema_segment_counts: np.ndarray,
    hma_segment_starts: np.ndarray,
    hma_segment_ends: np.ndarray,
    hma_segment_values: np.ndarray,
    hma_segment_counts: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open_1m: np.ndarray,
    exec_close_1m: np.ndarray,
    T_exec: np.int32,
    init_cash_quote: float,
    fixed_quote: float,
    fee_rate: float,
    slippage_rate: float,
    safe_profit_percent: float,
    use_fixed_quote: np.int8,
    use_profit_lock: np.int8,
    bars_per_year_exec: float,
    close_on_end: np.int8,
    direction_mode: np.int8,
    out_total_return_pct: np.ndarray,
    out_max_drawdown_pct: np.ndarray,
    out_return_over_max_drawdown: np.ndarray,
    out_profit_factor: np.ndarray,
    out_trade_count: np.ndarray,
    out_sharpe_trades: np.ndarray,
    out_win_rate_pct: np.ndarray,
    out_avg_trade_ret_pct: np.ndarray,
    out_avg_trade_exec_bars: np.ndarray,
    out_exposure_pct: np.ndarray,
) -> None:
    """Score two-indicator combos by merging compressed signal segments instead of scanning every bar."""
    K = combo_dema_idx.shape[0]

    for k in nb.prange(K):
        di = combo_dema_idx[k]
        hi = combo_hma_idx[k]
        available_quote = init_cash_quote
        safe_quote = 0.0
        equity = init_cash_quote
        peak_equity = equity
        max_drawdown_pct = 0.0
        gross_profit_quote = 0.0
        gross_loss_quote = 0.0
        closed_trade_count = np.int32(0)
        win_count = np.int32(0)
        sum_trade_return = 0.0
        sum_trade_return_squared = 0.0
        total_trade_return_pct = 0.0
        total_trade_exec_bars = 0.0
        exposure_bars = 0.0
        current_dir = np.int8(0)
        current_entry = np.int32(0)
        dema_segment_idx = 0
        hma_segment_idx = 0

        while dema_segment_idx < dema_segment_counts[di] and hma_segment_idx < hma_segment_counts[hi]:
            dema_start = dema_segment_starts[di, dema_segment_idx]
            dema_end = dema_segment_ends[di, dema_segment_idx]
            hma_start = hma_segment_starts[hi, hma_segment_idx]
            hma_end = hma_segment_ends[hi, hma_segment_idx]
            segment_start = dema_start if dema_start >= hma_start else hma_start
            segment_end = dema_end if dema_end <= hma_end else hma_end

            if segment_start < segment_end:
                raw_dir = consensus_dir2(
                    dema_segment_values[di, dema_segment_idx],
                    hma_segment_values[hi, hma_segment_idx],
                )
                dirn = apply_direction_mode(raw_dir, direction_mode)
                if dirn != 0 or (direction_mode == 1 and current_dir != 0):
                    entry_exec = sig_entry_exec_idx[segment_start]
                    if entry_exec >= T_exec:
                        break
                    if dirn == 0:
                        (
                            available_quote,
                            safe_quote,
                            equity,
                            peak_equity,
                            max_drawdown_pct,
                            gross_profit_quote,
                            gross_loss_quote,
                            closed_trade_count,
                            win_count,
                            sum_trade_return,
                            sum_trade_return_squared,
                            total_trade_return_pct,
                            total_trade_exec_bars,
                            exposure_bars,
                        ) = apply_no_risk_trade_to_state(
                            current_entry,
                            current_dir,
                            np.int32(entry_exec),
                            float(exec_open_1m[entry_exec]),
                            exec_open_1m,
                            available_quote,
                            safe_quote,
                            equity,
                            peak_equity,
                            max_drawdown_pct,
                            gross_profit_quote,
                            gross_loss_quote,
                            closed_trade_count,
                            win_count,
                            sum_trade_return,
                            sum_trade_return_squared,
                            total_trade_return_pct,
                            total_trade_exec_bars,
                            exposure_bars,
                            init_cash_quote,
                            fixed_quote,
                            fee_rate,
                            slippage_rate,
                            safe_profit_percent,
                            use_fixed_quote,
                            use_profit_lock,
                        )
                        current_dir = np.int8(0)
                        current_entry = np.int32(0)
                    elif current_dir == 0:
                        current_dir = dirn
                        current_entry = np.int32(entry_exec)
                    elif dirn != current_dir:
                        (
                            available_quote,
                            safe_quote,
                            equity,
                            peak_equity,
                            max_drawdown_pct,
                            gross_profit_quote,
                            gross_loss_quote,
                            closed_trade_count,
                            win_count,
                            sum_trade_return,
                            sum_trade_return_squared,
                            total_trade_return_pct,
                            total_trade_exec_bars,
                            exposure_bars,
                        ) = apply_no_risk_trade_to_state(
                            current_entry,
                            current_dir,
                            np.int32(entry_exec),
                            float(exec_open_1m[entry_exec]),
                            exec_open_1m,
                            available_quote,
                            safe_quote,
                            equity,
                            peak_equity,
                            max_drawdown_pct,
                            gross_profit_quote,
                            gross_loss_quote,
                            closed_trade_count,
                            win_count,
                            sum_trade_return,
                            sum_trade_return_squared,
                            total_trade_return_pct,
                            total_trade_exec_bars,
                            exposure_bars,
                            init_cash_quote,
                            fixed_quote,
                            fee_rate,
                            slippage_rate,
                            safe_profit_percent,
                            use_fixed_quote,
                            use_profit_lock,
                        )
                        current_dir = dirn
                        current_entry = np.int32(entry_exec)

            if dema_end == segment_end:
                dema_segment_idx += 1
            if hma_end == segment_end:
                hma_segment_idx += 1

        if current_dir != 0 and close_on_end == 1 and T_exec > 0:
            exit_exec_idx = np.int32(T_exec - 1)
            (
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
            ) = apply_no_risk_trade_to_state(
                current_entry,
                current_dir,
                exit_exec_idx,
                float(exec_close_1m[exit_exec_idx]),
                exec_open_1m,
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
                init_cash_quote,
                fixed_quote,
                fee_rate,
                slippage_rate,
                safe_profit_percent,
                use_fixed_quote,
                use_profit_lock,
            )

        total_return_pct = ((equity / init_cash_quote) - 1.0) * 100.0
        if gross_loss_quote > 0.0:
            profit_factor = gross_profit_quote / gross_loss_quote
        elif gross_profit_quote > 0.0:
            profit_factor = np.inf
        else:
            profit_factor = 0.0

        if max_drawdown_pct > 0.0:
            return_over_max_drawdown = total_return_pct / max_drawdown_pct
        elif total_return_pct > 0.0:
            return_over_max_drawdown = np.inf
        else:
            return_over_max_drawdown = 0.0

        if closed_trade_count > 0:
            win_rate_pct = (float(win_count) / float(closed_trade_count)) * 100.0
            avg_trade_ret_pct = total_trade_return_pct / float(closed_trade_count)
            avg_trade_exec_bars = total_trade_exec_bars / float(closed_trade_count)
        else:
            win_rate_pct = 0.0
            avg_trade_ret_pct = 0.0
            avg_trade_exec_bars = 0.0

        exposure_pct = (exposure_bars / float(T_exec)) * 100.0 if T_exec > 0 else 0.0
        sharpe_trades = trade_sharpe_kernel(
            closed_trade_count,
            sum_trade_return,
            sum_trade_return_squared,
            bars_per_year_exec,
            T_exec,
        )
        out_total_return_pct[k] = total_return_pct
        out_max_drawdown_pct[k] = max_drawdown_pct
        out_return_over_max_drawdown[k] = return_over_max_drawdown
        out_profit_factor[k] = profit_factor
        out_trade_count[k] = closed_trade_count
        out_sharpe_trades[k] = sharpe_trades
        out_win_rate_pct[k] = win_rate_pct
        out_avg_trade_ret_pct[k] = avg_trade_ret_pct
        out_avg_trade_exec_bars[k] = avg_trade_exec_bars
        out_exposure_pct[k] = exposure_pct


@njit_cached(parallel=True, fastmath=True)
def evaluate_no_risk_event_segments_n(
    combo_idx_by_indicator: np.ndarray,
    segment_starts: np.ndarray,
    segment_ends: np.ndarray,
    segment_values: np.ndarray,
    segment_counts: np.ndarray,
    segment_pos_workspace: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open_1m: np.ndarray,
    exec_close_1m: np.ndarray,
    T_exec: np.int32,
    init_cash_quote: float,
    fixed_quote: float,
    fee_rate: float,
    slippage_rate: float,
    safe_profit_percent: float,
    use_fixed_quote: np.int8,
    use_profit_lock: np.int8,
    bars_per_year_exec: float,
    close_on_end: np.int8,
    direction_mode: np.int8,
    out_total_return_pct: np.ndarray,
    out_max_drawdown_pct: np.ndarray,
    out_return_over_max_drawdown: np.ndarray,
    out_profit_factor: np.ndarray,
    out_trade_count: np.ndarray,
    out_sharpe_trades: np.ndarray,
    out_win_rate_pct: np.ndarray,
    out_avg_trade_ret_pct: np.ndarray,
    out_avg_trade_exec_bars: np.ndarray,
    out_exposure_pct: np.ndarray,
) -> None:
    """Score N-indicator no-risk combos by merging compressed signal segments.

    This is the generic backend behind the arity-specific slots 3..10. The 2-indicator
    path keeps its specialized kernel because it is the dominant current workload.
    """
    arity = combo_idx_by_indicator.shape[0]
    K = combo_idx_by_indicator.shape[1]

    for k in nb.prange(K):
        for indicator_pos in range(arity):
            segment_pos_workspace[k, indicator_pos] = np.int32(0)

        available_quote = init_cash_quote
        safe_quote = 0.0
        equity = init_cash_quote
        peak_equity = equity
        max_drawdown_pct = 0.0
        gross_profit_quote = 0.0
        gross_loss_quote = 0.0
        closed_trade_count = np.int32(0)
        win_count = np.int32(0)
        sum_trade_return = 0.0
        sum_trade_return_squared = 0.0
        total_trade_return_pct = 0.0
        total_trade_exec_bars = 0.0
        exposure_bars = 0.0
        current_dir = np.int8(0)
        current_entry = np.int32(0)

        while True:
            active = True
            segment_start = np.int32(0)
            segment_end = np.int32(2147483647)

            for indicator_pos in range(arity):
                row_idx = combo_idx_by_indicator[indicator_pos, k]
                segment_idx = segment_pos_workspace[k, indicator_pos]
                if segment_idx >= segment_counts[indicator_pos, row_idx]:
                    active = False
                    break
                start_value = segment_starts[indicator_pos, row_idx, segment_idx]
                end_value = segment_ends[indicator_pos, row_idx, segment_idx]
                if start_value > segment_start:
                    segment_start = start_value
                if end_value < segment_end:
                    segment_end = end_value

            if not active:
                break

            if segment_start < segment_end:
                first_row_idx = combo_idx_by_indicator[0, k]
                first_segment_idx = segment_pos_workspace[k, 0]
                raw_dir = segment_values[0, first_row_idx, first_segment_idx]
                if raw_dir != 0:
                    for indicator_pos in range(1, arity):
                        row_idx = combo_idx_by_indicator[indicator_pos, k]
                        segment_idx = segment_pos_workspace[k, indicator_pos]
                        if segment_values[indicator_pos, row_idx, segment_idx] != raw_dir:
                            raw_dir = np.int8(0)
                            break
                dirn = apply_direction_mode(raw_dir, direction_mode)

                if dirn != 0 or (direction_mode == 1 and current_dir != 0):
                    entry_exec = sig_entry_exec_idx[segment_start]
                    if entry_exec >= T_exec:
                        break
                    if dirn == 0:
                        (
                            available_quote,
                            safe_quote,
                            equity,
                            peak_equity,
                            max_drawdown_pct,
                            gross_profit_quote,
                            gross_loss_quote,
                            closed_trade_count,
                            win_count,
                            sum_trade_return,
                            sum_trade_return_squared,
                            total_trade_return_pct,
                            total_trade_exec_bars,
                            exposure_bars,
                        ) = apply_no_risk_trade_to_state(
                            current_entry,
                            current_dir,
                            np.int32(entry_exec),
                            float(exec_open_1m[entry_exec]),
                            exec_open_1m,
                            available_quote,
                            safe_quote,
                            equity,
                            peak_equity,
                            max_drawdown_pct,
                            gross_profit_quote,
                            gross_loss_quote,
                            closed_trade_count,
                            win_count,
                            sum_trade_return,
                            sum_trade_return_squared,
                            total_trade_return_pct,
                            total_trade_exec_bars,
                            exposure_bars,
                            init_cash_quote,
                            fixed_quote,
                            fee_rate,
                            slippage_rate,
                            safe_profit_percent,
                            use_fixed_quote,
                            use_profit_lock,
                        )
                        current_dir = np.int8(0)
                        current_entry = np.int32(0)
                    elif current_dir == 0:
                        current_dir = dirn
                        current_entry = np.int32(entry_exec)
                    elif dirn != current_dir:
                        (
                            available_quote,
                            safe_quote,
                            equity,
                            peak_equity,
                            max_drawdown_pct,
                            gross_profit_quote,
                            gross_loss_quote,
                            closed_trade_count,
                            win_count,
                            sum_trade_return,
                            sum_trade_return_squared,
                            total_trade_return_pct,
                            total_trade_exec_bars,
                            exposure_bars,
                        ) = apply_no_risk_trade_to_state(
                            current_entry,
                            current_dir,
                            np.int32(entry_exec),
                            float(exec_open_1m[entry_exec]),
                            exec_open_1m,
                            available_quote,
                            safe_quote,
                            equity,
                            peak_equity,
                            max_drawdown_pct,
                            gross_profit_quote,
                            gross_loss_quote,
                            closed_trade_count,
                            win_count,
                            sum_trade_return,
                            sum_trade_return_squared,
                            total_trade_return_pct,
                            total_trade_exec_bars,
                            exposure_bars,
                            init_cash_quote,
                            fixed_quote,
                            fee_rate,
                            slippage_rate,
                            safe_profit_percent,
                            use_fixed_quote,
                            use_profit_lock,
                        )
                        current_dir = dirn
                        current_entry = np.int32(entry_exec)

            for indicator_pos in range(arity):
                row_idx = combo_idx_by_indicator[indicator_pos, k]
                segment_idx = segment_pos_workspace[k, indicator_pos]
                if segment_ends[indicator_pos, row_idx, segment_idx] == segment_end:
                    segment_pos_workspace[k, indicator_pos] = np.int32(segment_idx + 1)

        if current_dir != 0 and close_on_end == 1 and T_exec > 0:
            exit_exec_idx = np.int32(T_exec - 1)
            (
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
            ) = apply_no_risk_trade_to_state(
                current_entry,
                current_dir,
                exit_exec_idx,
                float(exec_close_1m[exit_exec_idx]),
                exec_open_1m,
                available_quote,
                safe_quote,
                equity,
                peak_equity,
                max_drawdown_pct,
                gross_profit_quote,
                gross_loss_quote,
                closed_trade_count,
                win_count,
                sum_trade_return,
                sum_trade_return_squared,
                total_trade_return_pct,
                total_trade_exec_bars,
                exposure_bars,
                init_cash_quote,
                fixed_quote,
                fee_rate,
                slippage_rate,
                safe_profit_percent,
                use_fixed_quote,
                use_profit_lock,
            )

        total_return_pct = ((equity / init_cash_quote) - 1.0) * 100.0
        if gross_loss_quote > 0.0:
            profit_factor = gross_profit_quote / gross_loss_quote
        elif gross_profit_quote > 0.0:
            profit_factor = np.inf
        else:
            profit_factor = 0.0

        if max_drawdown_pct > 0.0:
            return_over_max_drawdown = total_return_pct / max_drawdown_pct
        elif total_return_pct > 0.0:
            return_over_max_drawdown = np.inf
        else:
            return_over_max_drawdown = 0.0

        if closed_trade_count > 0:
            win_rate_pct = (float(win_count) / float(closed_trade_count)) * 100.0
            avg_trade_ret_pct = total_trade_return_pct / float(closed_trade_count)
            avg_trade_exec_bars = total_trade_exec_bars / float(closed_trade_count)
        else:
            win_rate_pct = 0.0
            avg_trade_ret_pct = 0.0
            avg_trade_exec_bars = 0.0

        exposure_pct = (exposure_bars / float(T_exec)) * 100.0 if T_exec > 0 else 0.0
        sharpe_trades = trade_sharpe_kernel(
            closed_trade_count,
            sum_trade_return,
            sum_trade_return_squared,
            bars_per_year_exec,
            T_exec,
        )
        out_total_return_pct[k] = total_return_pct
        out_max_drawdown_pct[k] = max_drawdown_pct
        out_return_over_max_drawdown[k] = return_over_max_drawdown
        out_profit_factor[k] = profit_factor
        out_trade_count[k] = closed_trade_count
        out_sharpe_trades[k] = sharpe_trades
        out_win_rate_pct[k] = win_rate_pct
        out_avg_trade_ret_pct[k] = avg_trade_ret_pct
        out_avg_trade_exec_bars[k] = avg_trade_exec_bars
        out_exposure_pct[k] = exposure_pct


def evaluate_no_risk_trade_list_slow_two(
    combo_dema_idx: np.ndarray,
    combo_hma_idx: np.ndarray,
    dema_trade_T: np.ndarray,
    hma_trade_T: np.ndarray,
    sig_entry_exec_idx: np.ndarray,
    exec_open_1m: np.ndarray,
    exec_close_1m: np.ndarray,
    T_exec: np.int32,
):
    """Reference no-risk evaluation for a small subset of two-indicator combos.

    Parameters:
        combo_dema_idx: Local DEMA row indices.
        combo_hma_idx: Local HMA row indices.
        dema_trade_T: Filtered DEMA trade matrix.
        hma_trade_T: Filtered HMA trade matrix.
        sig_entry_exec_idx: 15m signal-bar to 1m execution-entry mapping.
        exec_open_1m: Full 1m open series.
        exec_close_1m: Full 1m close series.
        T_exec: Exclusive 1m execution limit for the run range.

    Returns:
        List of metric tuples aligned to the input combo order.

    Assumptions:
        Used only by the notebook self-check on a tiny subset.

    Raises:
        None.

    Side effects:
        None.
    """
    results = []
    n_sig = dema_trade_T.shape[1]
    for di, hi in zip(combo_dema_idx.tolist(), combo_hma_idx.tolist()):
        entry_arr = np.empty(n_sig, dtype=np.int32)
        dir_arr = np.empty(n_sig, dtype=np.int8)
        sig_exit_arr = np.empty(n_sig, dtype=np.int32)
        n_trades = int(build_trade_list_for_two_rows(
            dema_trade_T[int(di)],
            hma_trade_T[int(hi)],
            sig_entry_exec_idx,
            T_exec,
            entry_arr,
            dir_arr,
            sig_exit_arr,
        ))
        if n_trades <= 0:
            results.append((0.0, 0.0, 0.0, 0.0, 0, 0.0, 0.0, 0.0, 0.0, 0.0))
            continue
        results.append(
            score_trade_list_no_risk(
                entry_arr,
                dir_arr,
                sig_exit_arr,
                np.int32(n_trades),
                exec_open_1m,
                exec_close_1m,
                T_exec,
                INIT_CASH_QUOTE,
                FIXED_QUOTE,
                FEE_RATE,
                SLIPPAGE_RATE,
                SAFE_PROFIT_PERCENT,
                np.int8(1 if USE_FIXED_QUOTE else 0),
                np.int8(1 if USE_PROFIT_LOCK else 0),
                BARS_PER_YEAR_EXEC_1M,
                CLOSE_ON_END,
            )
        )
    return results


In [ ]:
EXACT_BACKEND_AUTO = "auto"
EXACT_BACKEND_EVENT_SEGMENTS_2_NO_RISK = "event_segments_2_no_risk"
EXACT_BACKEND_STREAMING_2_NO_RISK = "streaming_2_no_risk"
EXACT_BACKEND_GENERIC_EVENT_SEGMENTS_N_NO_RISK = "event_segments_n_no_risk"
EXACT_BACKEND_EVENT_SEGMENTS_N_NO_RISK_BY_ARITY = {
    arity: f"event_segments_{arity}_no_risk"
    for arity in range(1, 11)
}
EXACT_BACKEND_SUPPORTED_ARITIES = tuple(range(1, 11))


def resolve_no_risk_exact_backend(*, indicator_ids: tuple[str, ...], requested_backend: str = EXACT_BACKEND_AUTO) -> dict:
    """Resolve a no-risk exact backend strategy for the requested indicator arity."""
    indicator_ids = tuple(indicator_ids)
    arity = len(indicator_ids)
    if arity not in EXACT_BACKEND_SUPPORTED_ARITIES:
        raise NotImplementedError(
            f"No no-risk exact backend slot for arity={arity}. Supported slots: {EXACT_BACKEND_SUPPORTED_ARITIES}."
        )

    if arity == 2 and requested_backend in (EXACT_BACKEND_AUTO, "event_segments", EXACT_BACKEND_EVENT_SEGMENTS_2_NO_RISK):
        return {
            "name": EXACT_BACKEND_EVENT_SEGMENTS_2_NO_RISK,
            "implementation": EXACT_BACKEND_EVENT_SEGMENTS_2_NO_RISK,
            "mode": "no_risk",
            "arity": arity,
            "indicator_ids": indicator_ids,
            "requires_segments": True,
        }
    if arity == 2 and requested_backend in ("streaming", EXACT_BACKEND_STREAMING_2_NO_RISK):
        return {
            "name": EXACT_BACKEND_STREAMING_2_NO_RISK,
            "implementation": EXACT_BACKEND_STREAMING_2_NO_RISK,
            "mode": "no_risk",
            "arity": arity,
            "indicator_ids": indicator_ids,
            "requires_segments": False,
        }

    generic_slot = EXACT_BACKEND_EVENT_SEGMENTS_N_NO_RISK_BY_ARITY.get(arity)
    if arity != 2 and requested_backend in (
        EXACT_BACKEND_AUTO,
        "event_segments",
        EXACT_BACKEND_GENERIC_EVENT_SEGMENTS_N_NO_RISK,
        generic_slot,
    ):
        return {
            "name": generic_slot,
            "implementation": EXACT_BACKEND_GENERIC_EVENT_SEGMENTS_N_NO_RISK,
            "mode": "no_risk",
            "arity": arity,
            "indicator_ids": indicator_ids,
            "requires_segments": True,
        }

    raise NotImplementedError(
        f"No no-risk exact backend for arity={arity}, requested_backend={requested_backend!r}. "
        "Use auto/event_segments for supported arity slots, or add a specialized backend."
    )


def build_segment_stack(*, indicator_ids: tuple[str, ...], indicator_pools: dict) -> dict:
    """Pack per-indicator padded segments into an arity-first stack for generic exact kernels."""
    indicator_ids = tuple(indicator_ids)
    row_counts = [int(indicator_pools[indicator_id]["trade_T"].shape[0]) for indicator_id in indicator_ids]
    segment_widths = [int(indicator_pools[indicator_id]["segments"]["starts"].shape[1]) for indicator_id in indicator_ids]
    max_rows = max(row_counts)
    max_segments = max(segment_widths)
    arity = len(indicator_ids)

    starts = np.zeros((arity, max_rows, max_segments), dtype=np.int32)
    ends = np.zeros((arity, max_rows, max_segments), dtype=np.int32)
    values = np.zeros((arity, max_rows, max_segments), dtype=np.int8)
    counts = np.zeros((arity, max_rows), dtype=np.int32)

    for indicator_pos, indicator_id in enumerate(indicator_ids):
        segments = indicator_pools[indicator_id]["segments"]
        row_count = row_counts[indicator_pos]
        width = segment_widths[indicator_pos]
        starts[indicator_pos, :row_count, :width] = segments["starts"]
        ends[indicator_pos, :row_count, :width] = segments["ends"]
        values[indicator_pos, :row_count, :width] = segments["values"]
        counts[indicator_pos, :row_count] = segments["counts"]

    return {
        "indicator_ids": indicator_ids,
        "starts": np.ascontiguousarray(starts),
        "ends": np.ascontiguousarray(ends),
        "values": np.ascontiguousarray(values),
        "counts": np.ascontiguousarray(counts),
    }


def build_eval_stack(*, indicator_ids: tuple[str, ...], indicator_pools: dict) -> dict:
    """Pack eval_T rows into an arity-first stack for generic combo proxy scoring."""
    indicator_ids = tuple(indicator_ids)
    row_counts = [int(indicator_pools[indicator_id]["eval_T"].shape[0]) for indicator_id in indicator_ids]
    max_rows = max(row_counts)
    arity = len(indicator_ids)
    eval_stack = np.zeros((arity, max_rows, n_signal_intervals), dtype=np.int8)
    for indicator_pos, indicator_id in enumerate(indicator_ids):
        rows = row_counts[indicator_pos]
        eval_stack[indicator_pos, :rows, :] = indicator_pools[indicator_id]["eval_T"]
    return {
        "indicator_ids": indicator_ids,
        "eval_T": np.ascontiguousarray(eval_stack),
    }


def make_combo_idx_matrix(*, combo_chunk: dict, indicator_ids: tuple[str, ...]) -> np.ndarray:
    """Convert a chunk dict into an arity x K int32 matrix."""
    return np.ascontiguousarray(np.vstack([
        np.asarray(combo_chunk[indicator_id], dtype=np.int32)
        for indicator_id in indicator_ids
    ]))


def evaluate_no_risk_exact_chunk(
    *,
    selected: dict,
    indicator_pools: dict,
    exact_strategy: dict,
    exact_context: dict | None,
    total_return_pct: np.ndarray,
    max_drawdown_pct: np.ndarray,
    return_over_max_drawdown: np.ndarray,
    profit_factor: np.ndarray,
    trade_count: np.ndarray,
    sharpe_trades: np.ndarray,
    win_rate_pct: np.ndarray,
    avg_trade_ret_pct: np.ndarray,
    avg_trade_exec_bars: np.ndarray,
    exposure_pct: np.ndarray,
    direction_mode_code_value: np.int8,
) -> None:
    """Dispatch one no-risk exact chunk through the selected backend strategy."""
    indicator_ids = exact_strategy["indicator_ids"]
    arity = exact_strategy["arity"]
    if arity == 2 and exact_strategy["name"] == EXACT_BACKEND_EVENT_SEGMENTS_2_NO_RISK:
        left_id, right_id = indicator_ids
        left_segments = indicator_pools[left_id]["segments"]
        right_segments = indicator_pools[right_id]["segments"]
        evaluate_no_risk_event_segments_two(
            selected[left_id],
            selected[right_id],
            left_segments["starts"],
            left_segments["ends"],
            left_segments["values"],
            left_segments["counts"],
            right_segments["starts"],
            right_segments["ends"],
            right_segments["values"],
            right_segments["counts"],
            sig_entry_exec_idx_15m,
            price_fields_1m["open"],
            price_fields_1m["close"],
            T_exec_limit_1m,
            INIT_CASH_QUOTE,
            FIXED_QUOTE,
            FEE_RATE,
            SLIPPAGE_RATE,
            SAFE_PROFIT_PERCENT,
            np.int8(1 if USE_FIXED_QUOTE else 0),
            np.int8(1 if USE_PROFIT_LOCK else 0),
            BARS_PER_YEAR_EXEC_1M,
            CLOSE_ON_END,
            direction_mode_code_value,
            total_return_pct,
            max_drawdown_pct,
            return_over_max_drawdown,
            profit_factor,
            trade_count,
            sharpe_trades,
            win_rate_pct,
            avg_trade_ret_pct,
            avg_trade_exec_bars,
            exposure_pct,
        )
        return

    if arity == 2 and exact_strategy["name"] == EXACT_BACKEND_STREAMING_2_NO_RISK:
        left_id, right_id = indicator_ids
        evaluate_no_risk_streaming_two(
            selected[left_id],
            selected[right_id],
            indicator_pools[left_id]["trade_T"],
            indicator_pools[right_id]["trade_T"],
            sig_entry_exec_idx_15m,
            price_fields_1m["open"],
            price_fields_1m["close"],
            T_exec_limit_1m,
            INIT_CASH_QUOTE,
            FIXED_QUOTE,
            FEE_RATE,
            SLIPPAGE_RATE,
            SAFE_PROFIT_PERCENT,
            np.int8(1 if USE_FIXED_QUOTE else 0),
            np.int8(1 if USE_PROFIT_LOCK else 0),
            BARS_PER_YEAR_EXEC_1M,
            CLOSE_ON_END,
            total_return_pct,
            max_drawdown_pct,
            return_over_max_drawdown,
            profit_factor,
            trade_count,
            sharpe_trades,
            win_rate_pct,
            avg_trade_ret_pct,
            avg_trade_exec_bars,
            exposure_pct,
        )
        return

    if exact_strategy["implementation"] == EXACT_BACKEND_GENERIC_EVENT_SEGMENTS_N_NO_RISK:
        if exact_context is None:
            raise ValueError("Generic event-segment backend requires exact_context from build_segment_stack().")
        combo_idx_by_indicator = make_combo_idx_matrix(combo_chunk=selected, indicator_ids=indicator_ids)
        segment_pos_workspace = np.empty((combo_idx_by_indicator.shape[1], arity), dtype=np.int32)
        evaluate_no_risk_event_segments_n(
            combo_idx_by_indicator,
            exact_context["starts"],
            exact_context["ends"],
            exact_context["values"],
            exact_context["counts"],
            segment_pos_workspace,
            sig_entry_exec_idx_15m,
            price_fields_1m["open"],
            price_fields_1m["close"],
            T_exec_limit_1m,
            INIT_CASH_QUOTE,
            FIXED_QUOTE,
            FEE_RATE,
            SLIPPAGE_RATE,
            SAFE_PROFIT_PERCENT,
            np.int8(1 if USE_FIXED_QUOTE else 0),
            np.int8(1 if USE_PROFIT_LOCK else 0),
            BARS_PER_YEAR_EXEC_1M,
            CLOSE_ON_END,
            direction_mode_code_value,
            total_return_pct,
            max_drawdown_pct,
            return_over_max_drawdown,
            profit_factor,
            trade_count,
            sharpe_trades,
            win_rate_pct,
            avg_trade_ret_pct,
            avg_trade_exec_bars,
            exposure_pct,
        )
        return

    raise NotImplementedError(f"Unsupported exact backend: {exact_strategy['name']!r}.")


def evaluate_no_risk_exact_chunk_two(**kwargs) -> None:
    """Compatibility wrapper for older two-indicator call sites."""
    kwargs.setdefault("exact_context", None)
    evaluate_no_risk_exact_chunk(**kwargs)


@njit_cached(parallel=True, fastmath=True)
def proxy_prefilter_combos_chunk_two(
    combo_dema_idx: np.ndarray,
    combo_hma_idx: np.ndarray,
    dema_eval_T: np.ndarray,
    hma_eval_T: np.ndarray,
    ret_15m: np.ndarray,
    min_confirm: np.int32,
    fee_penalty_per_confirm: np.float32,
    out_confirm: np.ndarray,
    out_proxy: np.ndarray,
) -> None:
    """Compute cheap confirmation counts and proxy scores for one two-indicator combo chunk."""
    K = combo_dema_idx.shape[0]
    n_int = ret_15m.shape[0]

    for k in nb.prange(K):
        di = combo_dema_idx[k]
        hi = combo_hma_idx[k]
        confirms = np.int32(0)
        proxy = np.float32(0.0)

        for t in range(n_int):
            dirn = consensus_dir2(dema_eval_T[di, t], hma_eval_T[hi, t])
            if dirn == 1:
                confirms += 1
                proxy += ret_15m[t]
            elif dirn == -1:
                confirms += 1
                proxy -= ret_15m[t]

        out_confirm[k] = confirms
        if confirms >= min_confirm:
            out_proxy[k] = proxy - fee_penalty_per_confirm * np.float32(confirms)
        else:
            out_proxy[k] = NEG_INF


@njit_cached(parallel=True, fastmath=True)
def proxy_prefilter_combos_chunk_n(
    combo_idx_by_indicator: np.ndarray,
    eval_stack: np.ndarray,
    ret_15m: np.ndarray,
    min_confirm: np.int32,
    fee_penalty_per_confirm: np.float32,
    out_confirm: np.ndarray,
    out_proxy: np.ndarray,
) -> None:
    """Compute confirmation counts and proxy scores for generic N-indicator combo chunks."""
    arity = combo_idx_by_indicator.shape[0]
    K = combo_idx_by_indicator.shape[1]
    n_int = ret_15m.shape[0]

    for k in nb.prange(K):
        confirms = np.int32(0)
        proxy = np.float32(0.0)
        for t in range(n_int):
            first_row = combo_idx_by_indicator[0, k]
            dirn = eval_stack[0, first_row, t]
            if dirn == 0:
                continue
            for indicator_pos in range(1, arity):
                row_idx = combo_idx_by_indicator[indicator_pos, k]
                if eval_stack[indicator_pos, row_idx, t] != dirn:
                    dirn = np.int8(0)
                    break
            if dirn == 1:
                confirms += 1
                proxy += ret_15m[t]
            elif dirn == -1:
                confirms += 1
                proxy -= ret_15m[t]

        out_confirm[k] = confirms
        if confirms >= min_confirm:
            out_proxy[k] = proxy - fee_penalty_per_confirm * np.float32(confirms)
        else:
            out_proxy[k] = NEG_INF


@njit_cached(inline="always")
def proxy_for_two_rows(
    dema_eval_row: np.ndarray,
    hma_eval_row: np.ndarray,
    ret_15m: np.ndarray,
    min_confirm: np.int32,
    fee_penalty_per_confirm: np.float32,
) -> tuple[np.int32, np.float32]:
    """Compute confirm count and proxy score for one final two-indicator result."""
    confirms = np.int32(0)
    proxy = np.float32(0.0)
    for t in range(ret_15m.shape[0]):
        dirn = consensus_dir2(dema_eval_row[t], hma_eval_row[t])
        if dirn == 1:
            confirms += 1
            proxy += ret_15m[t]
        elif dirn == -1:
            confirms += 1
            proxy -= ret_15m[t]
    if confirms >= min_confirm:
        return confirms, proxy - fee_penalty_per_confirm * np.float32(confirms)
    return confirms, NEG_INF


def proxy_for_indicator_rows(*, eval_rows: tuple[np.ndarray, ...], ret_15m: np.ndarray, min_confirm: int, fee_penalty_per_confirm: np.float32) -> tuple[int, float]:
    """Compute confirm count and proxy score for a final result with arbitrary arity."""
    if len(eval_rows) == 2:
        confirms, proxy = proxy_for_two_rows(
            eval_rows[0],
            eval_rows[1],
            ret_15m,
            np.int32(min_confirm),
            fee_penalty_per_confirm,
        )
        return int(confirms), float(proxy)

    consensus = np.asarray(eval_rows[0], dtype=np.int8).copy()
    for eval_row in eval_rows[1:]:
        consensus[consensus != eval_row] = np.int8(0)
    confirms = int(np.count_nonzero(consensus))
    if confirms < int(min_confirm):
        return confirms, float(NEG_INF)
    proxy = np.float32(0.0)
    if confirms:
        proxy += np.sum(ret_15m[consensus == 1], dtype=np.float32)
        proxy -= np.sum(ret_15m[consensus == -1], dtype=np.float32)
    proxy -= fee_penalty_per_confirm * np.float32(confirms)
    return confirms, float(proxy)


def build_combo_proxy_cache_two(*, left_eval_T: np.ndarray, right_eval_T: np.ndarray, ret_15m: np.ndarray, min_confirm: int, fee_penalty_per_confirm: np.float32):
    """Build matrix-backed confirm/proxy lookup tables for active two-indicator pruning."""
    ret = ret_15m.astype(np.float32, copy=False)
    left_pos = (left_eval_T == 1).astype(np.float32)
    left_neg = (left_eval_T == -1).astype(np.float32)
    right_pos = (right_eval_T == 1).astype(np.float32)
    right_neg = (right_eval_T == -1).astype(np.float32)

    proxy_matrix = left_pos @ np.ascontiguousarray((right_pos * ret).T)
    proxy_matrix -= left_neg @ np.ascontiguousarray((right_neg * ret).T)
    confirm_matrix = left_pos @ np.ascontiguousarray(right_pos.T)
    confirm_matrix += left_neg @ np.ascontiguousarray(right_neg.T)
    confirm_matrix = np.rint(confirm_matrix).astype(np.int32)
    proxy_matrix = proxy_matrix.astype(np.float32, copy=False)
    proxy_matrix -= fee_penalty_per_confirm * confirm_matrix.astype(np.float32)
    proxy_matrix[confirm_matrix < int(min_confirm)] = NEG_INF
    return confirm_matrix, proxy_matrix


def gather_combo_proxy_cache_two(*, combo_chunk: dict, combo_proxy_cache, indicator_ids: tuple[str, ...]):
    """Gather confirm/proxy vectors for one two-indicator chunk from matrix-backed lookup tables."""
    confirm_matrix, proxy_matrix = combo_proxy_cache
    left_id, right_id = indicator_ids
    left_idx = combo_chunk[left_id]
    right_idx = combo_chunk[right_id]
    return (
        np.ascontiguousarray(confirm_matrix[left_idx, right_idx]),
        np.ascontiguousarray(proxy_matrix[left_idx, right_idx]),
    )


def iter_combo_chunks(*, indicator_ids: tuple[str, ...], local_row_pools: dict, chunk_size: int):
    """Yield bounded Cartesian combo chunks over filtered local row pools for any supported arity."""
    indicator_ids = tuple(indicator_ids)
    buffers = {indicator_id: [] for indicator_id in indicator_ids}

    for combo in itertools.product(*(local_row_pools[indicator_id] for indicator_id in indicator_ids)):
        for indicator_id, value in zip(indicator_ids, combo):
            buffers[indicator_id].append(int(value))
        if len(buffers[indicator_ids[0]]) >= chunk_size:
            yield {indicator_id: np.asarray(buffers[indicator_id], dtype=np.int32) for indicator_id in indicator_ids}
            buffers = {indicator_id: [] for indicator_id in indicator_ids}

    if buffers[indicator_ids[0]]:
        yield {indicator_id: np.asarray(buffers[indicator_id], dtype=np.int32) for indicator_id in indicator_ids}


def iter_combo_chunks_two(*, local_row_pools, chunk_size: int):
    """Compatibility wrapper over the generic chunk iterator for the default two-indicator search."""
    return iter_combo_chunks(indicator_ids=("ma.dema", "ma.hma"), local_row_pools=local_row_pools, chunk_size=chunk_size)


def run_fast_vs_reference_self_check_two(*, combo_chunk, indicator_pools, exact_strategy: dict, exact_context: dict | None = None, check_n: int = SELF_CHECK_N_DEFAULT, ret_tol: float = 1e-4):
    """Compare the selected two-indicator exact backend against the slow trade-list reference."""
    if exact_strategy["arity"] != 2:
        return {
            "checked": 0,
            "exact_backend": exact_strategy["name"],
            "reason": f"slow reference is currently defined only for arity=2, got arity={exact_strategy['arity']}",
        }

    left_id, right_id = exact_strategy["indicator_ids"]
    n_check = min(check_n, int(combo_chunk[left_id].shape[0]))
    if n_check <= 0:
        return {"checked": 0, "max_abs_best_ret_diff": 0.0, "max_abs_exact_backend_ret_diff": 0.0}

    subset = {indicator_id: combo_chunk[indicator_id][:n_check] for indicator_id in exact_strategy["indicator_ids"]}
    trade_counts = np.empty(n_check, dtype=np.int32)
    count_trades_for_two_combos(
        subset[left_id],
        subset[right_id],
        indicator_pools[left_id]["trade_T"],
        indicator_pools[right_id]["trade_T"],
        sig_entry_exec_idx_15m,
        T_exec_limit_1m,
        trade_counts,
    )

    fast_total_return_pct = np.empty(n_check, dtype=np.float64)
    fast_max_drawdown_pct = np.empty(n_check, dtype=np.float64)
    fast_return_over_max_drawdown = np.empty(n_check, dtype=np.float64)
    fast_profit_factor = np.empty(n_check, dtype=np.float64)
    fast_trade_count = np.empty(n_check, dtype=np.int32)
    fast_sharpe_trades = np.empty(n_check, dtype=np.float64)
    fast_win_rate_pct = np.empty(n_check, dtype=np.float64)
    fast_avg_trade_ret_pct = np.empty(n_check, dtype=np.float64)
    fast_avg_trade_exec_bars = np.empty(n_check, dtype=np.float64)
    fast_exposure_pct = np.empty(n_check, dtype=np.float64)

    evaluate_no_risk_trade_list_fast_two(
        subset[left_id],
        subset[right_id],
        indicator_pools[left_id]["trade_T"],
        indicator_pools[right_id]["trade_T"],
        sig_entry_exec_idx_15m,
        price_fields_1m["open"],
        price_fields_1m["close"],
        T_exec_limit_1m,
        trade_counts,
        INIT_CASH_QUOTE,
        FIXED_QUOTE,
        FEE_RATE,
        SLIPPAGE_RATE,
        SAFE_PROFIT_PERCENT,
        np.int8(1 if USE_FIXED_QUOTE else 0),
        np.int8(1 if USE_PROFIT_LOCK else 0),
        BARS_PER_YEAR_EXEC_1M,
        CLOSE_ON_END,
        fast_total_return_pct,
        fast_max_drawdown_pct,
        fast_return_over_max_drawdown,
        fast_profit_factor,
        fast_trade_count,
        fast_sharpe_trades,
        fast_win_rate_pct,
        fast_avg_trade_ret_pct,
        fast_avg_trade_exec_bars,
        fast_exposure_pct,
    )

    slow = evaluate_no_risk_trade_list_slow_two(
        subset[left_id],
        subset[right_id],
        indicator_pools[left_id]["trade_T"],
        indicator_pools[right_id]["trade_T"],
        sig_entry_exec_idx_15m,
        price_fields_1m["open"],
        price_fields_1m["close"],
        T_exec_limit_1m,
    )
    slow_total_return_pct = np.asarray([row[0] for row in slow], dtype=np.float64)
    slow_trade_count = np.asarray([row[4] for row in slow], dtype=np.int32)

    if not np.array_equal(slow_trade_count, fast_trade_count):
        raise AssertionError("Fast trade-list counts differ from the slow reference.")

    max_abs_best_ret_diff = float(np.max(np.abs(slow_total_return_pct - fast_total_return_pct)))
    if max_abs_best_ret_diff > ret_tol:
        raise AssertionError(
            f"Fast trade-list total return differs from the slow reference by {max_abs_best_ret_diff}, tolerance {ret_tol}."
        )

    backend_total_return_pct = np.empty(n_check, dtype=np.float64)
    backend_max_drawdown_pct = np.empty(n_check, dtype=np.float64)
    backend_return_over_max_drawdown = np.empty(n_check, dtype=np.float64)
    backend_profit_factor = np.empty(n_check, dtype=np.float64)
    backend_trade_count = np.empty(n_check, dtype=np.int32)
    backend_sharpe_trades = np.empty(n_check, dtype=np.float64)
    backend_win_rate_pct = np.empty(n_check, dtype=np.float64)
    backend_avg_trade_ret_pct = np.empty(n_check, dtype=np.float64)
    backend_avg_trade_exec_bars = np.empty(n_check, dtype=np.float64)
    backend_exposure_pct = np.empty(n_check, dtype=np.float64)
    evaluate_no_risk_exact_chunk(
        selected=subset,
        indicator_pools=indicator_pools,
        exact_strategy=exact_strategy,
        exact_context=exact_context,
        total_return_pct=backend_total_return_pct,
        max_drawdown_pct=backend_max_drawdown_pct,
        return_over_max_drawdown=backend_return_over_max_drawdown,
        profit_factor=backend_profit_factor,
        trade_count=backend_trade_count,
        sharpe_trades=backend_sharpe_trades,
        win_rate_pct=backend_win_rate_pct,
        avg_trade_ret_pct=backend_avg_trade_ret_pct,
        avg_trade_exec_bars=backend_avg_trade_exec_bars,
        exposure_pct=backend_exposure_pct,
    )
    if not np.array_equal(slow_trade_count, backend_trade_count):
        raise AssertionError(f"Exact backend {exact_strategy['name']!r} trade counts differ from the slow reference.")
    max_abs_exact_backend_ret_diff = float(np.max(np.abs(slow_total_return_pct - backend_total_return_pct)))
    if max_abs_exact_backend_ret_diff > ret_tol:
        raise AssertionError(
            f"Exact backend {exact_strategy['name']!r} total return differs from the slow reference by "
            f"{max_abs_exact_backend_ret_diff}, tolerance {ret_tol}."
        )

    return {
        "checked": n_check,
        "exact_backend": exact_strategy["name"],
        "max_abs_best_ret_diff": max_abs_best_ret_diff,
        "max_abs_exact_backend_ret_diff": max_abs_exact_backend_ret_diff,
    }


def build_combo_proxy_context(*, indicator_ids: tuple[str, ...], indicator_pools: dict, combo_prefilter_active: bool, combo_min_confirm: int, fee_penalty_per_confirm: np.float32) -> dict | None:
    """Build the active proxy context for combo pruning, or return None when pruning is disabled."""
    if not combo_prefilter_active:
        return None
    if len(indicator_ids) == 2:
        left_id, right_id = indicator_ids
        return {
            "type": "matrix_two",
            "cache": build_combo_proxy_cache_two(
                left_eval_T=indicator_pools[left_id]["eval_T"],
                right_eval_T=indicator_pools[right_id]["eval_T"],
                ret_15m=signal_returns_15m,
                min_confirm=combo_min_confirm,
                fee_penalty_per_confirm=fee_penalty_per_confirm,
            ),
        }
    return {
        "type": "generic_n",
        "eval_stack": build_eval_stack(indicator_ids=indicator_ids, indicator_pools=indicator_pools),
    }


def search_topk_indicator_no_risk(
    *,
    indicator_ids: tuple[str, ...],
    row_pools: dict,
    indicator_top_frac: float = PREFILTER_TOP_FRAC,
    min_nonzero: int = PREFILTER_MIN_NONZERO,
    combo_top_frac: float = COMBO_PREFILTER_TOP_FRAC,
    combo_min_confirm: int = COMBO_MIN_CONFIRM,
    combo_chunk_size: int = COMBO_CHUNK_SIZE,
    top_k: int = TOP_K_DEFAULT,
    self_check_n: int = SELF_CHECK_N_DEFAULT,
    exact_backend: str = EXACT_BACKEND_AUTO,
    max_cartesian_candidates: int | None = None,
    direction_mode: str = DIRECTION_MODE_LONG_SHORT_REVERSAL,
    verbose: bool = True,
):
    """Run a no-risk search for any supported indicator arity using standard pools and backend slots."""
    indicator_ids = tuple(indicator_ids)
    direction_mode_code_value = direction_mode_code(direction_mode)
    total_start = time.perf_counter()
    timers = {
        "service_warmup": 0.0,
        "numba_warmup": 0.0,
        "prepare_pools": 0.0,
        "build_exact_context": 0.0,
        "build_proxy_context": 0.0,
        "combo_iteration": 0.0,
        "proxy_filter": 0.0,
        "self_check": 0.0,
        "exact_scoring": 0.0,
        "heap_update": 0.0,
        "top_result_proxy_fill": 0.0,
        "total_without_warmup": 0.0,
        "total": 0.0,
    }

    exact_strategy = resolve_no_risk_exact_backend(indicator_ids=indicator_ids, requested_backend=exact_backend)
    missing_pools = [indicator_id for indicator_id in indicator_ids if indicator_id not in row_pools]
    if missing_pools:
        raise KeyError(f"row_pools missing indicator ids: {missing_pools}")

    t0 = time.perf_counter()
    indicator_pools = prepare_indicator_pools(
        indicator_ids=indicator_ids,
        row_pools=row_pools,
        top_frac=indicator_top_frac,
        min_nonzero=min_nonzero,
        fee_rate=FEE_RATE,
        time_chunk=TIME_CHUNK,
    )
    timers["prepare_pools"] += time.perf_counter() - t0

    local_row_pools = {
        indicator_id: np.arange(indicator_pools[indicator_id]["trade_T"].shape[0], dtype=np.int32)
        for indicator_id in indicator_ids
    }
    filtered_pool_sizes = {
        indicator_id: int(indicator_pools[indicator_id]["trade_T"].shape[0])
        for indicator_id in indicator_ids
    }
    cartesian_size = int(math.prod(filtered_pool_sizes.values()))
    if max_cartesian_candidates is not None and cartesian_size > int(max_cartesian_candidates):
        raise ValueError(
            f"Cartesian search would evaluate {cartesian_size} combinations after row prefilter, "
            f"above max_cartesian_candidates={max_cartesian_candidates}."
        )

    exact_context = None
    if exact_strategy["implementation"] == EXACT_BACKEND_GENERIC_EVENT_SEGMENTS_N_NO_RISK:
        t0 = time.perf_counter()
        exact_context = build_segment_stack(indicator_ids=indicator_ids, indicator_pools=indicator_pools)
        timers["build_exact_context"] += time.perf_counter() - t0

    heap = []
    self_check = None
    total_combo_chunks = 0
    total_exact_candidates = 0
    fee_penalty_per_confirm = np.float32(1.5 * FEE_RATE)
    combo_prefilter_active = combo_top_frac < 1.0 or combo_min_confirm > 1

    t0 = time.perf_counter()
    combo_proxy_context = build_combo_proxy_context(
        indicator_ids=indicator_ids,
        indicator_pools=indicator_pools,
        combo_prefilter_active=combo_prefilter_active,
        combo_min_confirm=combo_min_confirm,
        fee_penalty_per_confirm=fee_penalty_per_confirm,
    )
    timers["build_proxy_context"] += time.perf_counter() - t0

    combo_iter = iter_combo_chunks(indicator_ids=indicator_ids, local_row_pools=local_row_pools, chunk_size=combo_chunk_size)
    while True:
        t0 = time.perf_counter()
        try:
            combo_chunk = next(combo_iter)
        except StopIteration:
            timers["combo_iteration"] += time.perf_counter() - t0
            break
        timers["combo_iteration"] += time.perf_counter() - t0

        total_combo_chunks += 1
        chunk_len = int(combo_chunk[indicator_ids[0]].shape[0])

        t0 = time.perf_counter()
        if not combo_prefilter_active:
            keep_idx = np.arange(chunk_len, dtype=np.int32)
            selected = {indicator_id: combo_chunk[indicator_id] for indicator_id in indicator_ids}
            selected_confirm = None
            selected_proxy = None
        else:
            out_confirm = np.empty(chunk_len, dtype=np.int32)
            out_proxy = np.empty(chunk_len, dtype=np.float32)
            if combo_proxy_context["type"] == "matrix_two":
                out_confirm, out_proxy = gather_combo_proxy_cache_two(
                    combo_chunk=combo_chunk,
                    combo_proxy_cache=combo_proxy_context["cache"],
                    indicator_ids=indicator_ids,
                )
            else:
                combo_idx_by_indicator = make_combo_idx_matrix(combo_chunk=combo_chunk, indicator_ids=indicator_ids)
                proxy_prefilter_combos_chunk_n(
                    combo_idx_by_indicator,
                    combo_proxy_context["eval_stack"]["eval_T"],
                    signal_returns_15m,
                    np.int32(combo_min_confirm),
                    fee_penalty_per_confirm,
                    out_confirm,
                    out_proxy,
                )
            valid_idx = np.flatnonzero(out_proxy > NEG_INF / 2)
            if valid_idx.size == 0:
                timers["proxy_filter"] += time.perf_counter() - t0
                continue
            keep_local = topk_fraction_idx(out_proxy[valid_idx], combo_top_frac)
            keep_idx = np.sort(valid_idx[keep_local].astype(np.int32))
            selected = {indicator_id: combo_chunk[indicator_id][keep_idx] for indicator_id in indicator_ids}
            selected_confirm = out_confirm[keep_idx]
            selected_proxy = out_proxy[keep_idx]
        timers["proxy_filter"] += time.perf_counter() - t0
        total_exact_candidates += int(keep_idx.size)

        if self_check is None and self_check_n > 0:
            t0 = time.perf_counter()
            self_check = run_fast_vs_reference_self_check_two(
                combo_chunk=selected,
                indicator_pools=indicator_pools,
                exact_strategy=exact_strategy,
                exact_context=exact_context,
                check_n=self_check_n,
                direction_mode=direction_mode,
            )
            timers["self_check"] += time.perf_counter() - t0

        result_size = int(keep_idx.size)
        total_return_pct = np.empty(result_size, dtype=np.float64)
        max_drawdown_pct = np.empty(result_size, dtype=np.float64)
        return_over_max_drawdown = np.empty(result_size, dtype=np.float64)
        profit_factor = np.empty(result_size, dtype=np.float64)
        trade_count = np.empty(result_size, dtype=np.int32)
        sharpe_trades = np.empty(result_size, dtype=np.float64)
        win_rate_pct = np.empty(result_size, dtype=np.float64)
        avg_trade_ret_pct = np.empty(result_size, dtype=np.float64)
        avg_trade_exec_bars = np.empty(result_size, dtype=np.float64)
        exposure_pct = np.empty(result_size, dtype=np.float64)

        t0 = time.perf_counter()
        evaluate_no_risk_exact_chunk(
            selected=selected,
            indicator_pools=indicator_pools,
            exact_strategy=exact_strategy,
            exact_context=exact_context,
            total_return_pct=total_return_pct,
            max_drawdown_pct=max_drawdown_pct,
            return_over_max_drawdown=return_over_max_drawdown,
            profit_factor=profit_factor,
            trade_count=trade_count,
            sharpe_trades=sharpe_trades,
            win_rate_pct=win_rate_pct,
            avg_trade_ret_pct=avg_trade_ret_pct,
            avg_trade_exec_bars=avg_trade_exec_bars,
            exposure_pct=exposure_pct,
            direction_mode_code_value=direction_mode_code_value,
        )
        timers["exact_scoring"] += time.perf_counter() - t0

        t0 = time.perf_counter()
        for local_idx in range(result_size):
            score = float(total_return_pct[local_idx])
            local_indices = tuple(int(selected[indicator_id][local_idx]) for indicator_id in indicator_ids)
            orig_rows = tuple(int(indicator_pools[indicator_id]["row_ids"][local_indices[pos]]) for pos, indicator_id in enumerate(indicator_ids))
            proxy_pending = selected_confirm is None
            if proxy_pending:
                confirm_count = 0
                proxy_score = 0.0
            else:
                confirm_count = int(selected_confirm[local_idx])
                proxy_score = float(selected_proxy[local_idx])
            item = {
                "total_return_pct": score,
                "confirm_count": confirm_count,
                "proxy_score": proxy_score,
                "trade_count": int(trade_count[local_idx]),
                "max_drawdown_pct": float(max_drawdown_pct[local_idx]),
                "return_over_max_drawdown": float(return_over_max_drawdown[local_idx]),
                "profit_factor": float(profit_factor[local_idx]),
                "sharpe_trades": float(sharpe_trades[local_idx]),
                "win_rate_pct": float(win_rate_pct[local_idx]),
                "avg_trade_ret_pct": float(avg_trade_ret_pct[local_idx]),
                "avg_trade_exec_bars": float(avg_trade_exec_bars[local_idx]),
                "exposure_pct": float(exposure_pct[local_idx]),
                "_local_indices": local_indices,
                "_proxy_pending": proxy_pending,
            }
            for pos, indicator_id in enumerate(indicator_ids):
                item[indicator_id] = indicator_pools[indicator_id]["metadata"][local_indices[pos]]
            heap_key = (score, orig_rows)
            heap_item = (heap_key, item)
            if len(heap) < top_k:
                heapq.heappush(heap, heap_item)
            elif heap_key > heap[0][0]:
                heapq.heapreplace(heap, heap_item)
        timers["heap_update"] += time.perf_counter() - t0

    if verbose:
        print("indicator ids:", indicator_ids)
        print("filtered pool sizes:", filtered_pool_sizes)
        print("cartesian combinations:", cartesian_size)
        print("combo chunks processed:", total_combo_chunks)
        print("exact candidates evaluated:", total_exact_candidates)
        print("exact backend:", exact_strategy["name"])
        if self_check is not None:
            print("self-check:", self_check)

    t0 = time.perf_counter()
    top_results = []
    for _, item in sorted(heap, key=lambda pair: pair[0], reverse=True):
        local_indices = item.pop("_local_indices")
        if item.pop("_proxy_pending"):
            eval_rows = tuple(
                indicator_pools[indicator_id]["eval_T"][local_indices[pos]]
                for pos, indicator_id in enumerate(indicator_ids)
            )
            confirm_count, proxy_score = proxy_for_indicator_rows(
                eval_rows=eval_rows,
                ret_15m=signal_returns_15m,
                min_confirm=combo_min_confirm,
                fee_penalty_per_confirm=fee_penalty_per_confirm,
            )
            item["confirm_count"] = int(confirm_count)
            item["proxy_score"] = float(proxy_score)
        top_results.append(item)
    timers["top_result_proxy_fill"] += time.perf_counter() - t0
    timers["total_without_warmup"] = time.perf_counter() - total_start
    timers["total"] = timers["total_without_warmup"]

    return {
        "indicator_ids": indicator_ids,
        "direction_mode": direction_mode,
        "filtered_pool_sizes": filtered_pool_sizes,
        "cartesian_combinations": cartesian_size,
        "combo_chunks_processed": total_combo_chunks,
        "exact_candidates_evaluated": total_exact_candidates,
        "indicator_pool_schema": INDICATOR_POOL_SCHEMA_KEYS,
        "self_check": self_check,
        "exact_backend": exact_strategy,
        "exact_engine": exact_strategy["name"],
        "timers": timers,
        "top_results": top_results,
    }


def search_topk_two_indicator_no_risk(*, row_pools, indicator_top_frac: float = PREFILTER_TOP_FRAC, min_nonzero: int = PREFILTER_MIN_NONZERO, combo_top_frac: float = COMBO_PREFILTER_TOP_FRAC, combo_min_confirm: int = COMBO_MIN_CONFIRM, combo_chunk_size: int = COMBO_CHUNK_SIZE, top_k: int = TOP_K_DEFAULT, self_check_n: int = SELF_CHECK_N_DEFAULT, exact_backend: str = EXACT_BACKEND_AUTO, direction_mode: str = DIRECTION_MODE_LONG_SHORT_REVERSAL, verbose: bool = True):
    """Compatibility wrapper for the default two-indicator no-risk search."""
    indicator_ids = tuple(row_pools.keys())
    if len(indicator_ids) != 2:
        indicator_ids = ("ma.dema", "ma.hma")
    return search_topk_indicator_no_risk(
        indicator_ids=indicator_ids,
        row_pools=row_pools,
        indicator_top_frac=indicator_top_frac,
        min_nonzero=min_nonzero,
        combo_top_frac=combo_top_frac,
        combo_min_confirm=combo_min_confirm,
        combo_chunk_size=combo_chunk_size,
        top_k=top_k,
        self_check_n=self_check_n,
        exact_backend=exact_backend,
        direction_mode=direction_mode,
        verbose=verbose,
    )


In [ ]:

# Production prototype extensions: generic parity, TP/SL risk-on path, benchmark matrix.

BACKEND_REGISTRY = {
    "event_segments_2_no_risk": {
        "risk_mode": "none",
        "arity": (2,),
        "default": True,
        "description": "Specialized 2-indicator event-compressed no-risk exact backend.",
    },
    "streaming_2_no_risk": {
        "risk_mode": "none",
        "arity": (2,),
        "default": False,
        "description": "Specialized 2-indicator streaming fallback for parity/perf comparison.",
    },
    "event_segments_n_no_risk": {
        "risk_mode": "none",
        "arity": tuple(range(1, 11)),
        "default": "arity != 2",
        "description": "Generic event-compressed no-risk exact backend for arity 1 and 3..10.",
    },
    "event_segments_n_tp_sl_15m_grid": {
        "risk_mode": "tp_sl_grid",
        "arity": tuple(range(1, 8)),
        "default": True,
        "description": "Generic event-compressed TP/SL backend backed by hit_times/15m.",
    },
}


def apply_direction_mode_py(raw_dir: int, direction_mode: str) -> int:
    """Map raw consensus direction into the requested trading direction mode for slow references."""
    if direction_mode == DIRECTION_MODE_LONG_ONLY:
        return 1 if raw_dir == 1 else 0
    if direction_mode == DIRECTION_MODE_LONG_SHORT_REVERSAL:
        return int(raw_dir)
    raise ValueError(f"Unsupported direction_mode={direction_mode!r}.")


def build_trade_list_for_indicator_rows_slow(*, indicator_ids: tuple[str, ...], indicator_pools: dict, local_indices: tuple[int, ...], direction_mode: str = DIRECTION_MODE_LONG_SHORT_REVERSAL):
    """Build a compact trade list for any arity by direct Python consensus scanning."""
    rows = [indicator_pools[indicator_id]["trade_T"][local_indices[pos]] for pos, indicator_id in enumerate(indicator_ids)]
    n_sig = rows[0].shape[0]
    entry_exec = []
    directions = []
    sig_exit_exec = []
    current_dir = 0
    current_entry = 0
    for t in range(n_sig):
        raw_dir = int(rows[0][t])
        if raw_dir != 0:
            for row in rows[1:]:
                if int(row[t]) != raw_dir:
                    raw_dir = 0
                    break
        dirn = apply_direction_mode_py(raw_dir, direction_mode)
        if dirn == 0 and not (direction_mode == DIRECTION_MODE_LONG_ONLY and current_dir != 0):
            continue
        entry_idx = int(sig_entry_exec_idx_15m[t])
        if entry_idx >= int(T_exec_limit_1m):
            break
        if dirn == 0:
            entry_exec.append(current_entry)
            directions.append(current_dir)
            sig_exit_exec.append(entry_idx)
            current_dir = 0
            current_entry = 0
            continue
        if current_dir == 0:
            current_dir = dirn
            current_entry = entry_idx
            continue
        if dirn == current_dir:
            continue
        entry_exec.append(current_entry)
        directions.append(current_dir)
        sig_exit_exec.append(entry_idx)
        current_dir = dirn
        current_entry = entry_idx
    if current_dir != 0:
        entry_exec.append(current_entry)
        directions.append(current_dir)
        sig_exit_exec.append(int(T_exec_limit_1m))
    return (
        np.asarray(entry_exec, dtype=np.int32),
        np.asarray(directions, dtype=np.int8),
        np.asarray(sig_exit_exec, dtype=np.int32),
    )



def evaluate_no_risk_reference_rows_slow(*, indicator_ids: tuple[str, ...], indicator_pools: dict, local_indices: tuple[int, ...], direction_mode: str = DIRECTION_MODE_LONG_SHORT_REVERSAL):
    """Reference no-risk scorer for a single arbitrary-arity combo."""
    entry_arr, dir_arr, exit_arr = build_trade_list_for_indicator_rows_slow(
        indicator_ids=indicator_ids,
        indicator_pools=indicator_pools,
        local_indices=local_indices,
        direction_mode=direction_mode,
    )
    if entry_arr.size == 0:
        return (0.0, 0.0, 0.0, 0.0, 0, 0.0, 0.0, 0.0, 0.0, 0.0)
    return score_trade_list_no_risk(
        entry_arr,
        dir_arr,
        exit_arr,
        np.int32(entry_arr.size),
        price_fields_1m["open"],
        price_fields_1m["close"],
        T_exec_limit_1m,
        INIT_CASH_QUOTE,
        FIXED_QUOTE,
        FEE_RATE,
        SLIPPAGE_RATE,
        SAFE_PROFIT_PERCENT,
        np.int8(1 if USE_FIXED_QUOTE else 0),
        np.int8(1 if USE_PROFIT_LOCK else 0),
        BARS_PER_YEAR_EXEC_1M,
        CLOSE_ON_END,
    )



def run_fast_vs_reference_self_check_two(*, combo_chunk, indicator_pools, exact_strategy: dict, exact_context: dict | None = None, check_n: int = SELF_CHECK_N_DEFAULT, ret_tol: float = 1e-4, direction_mode: str = DIRECTION_MODE_LONG_SHORT_REVERSAL):
    """Compare the selected no-risk exact backend against a generic slow reference for arity 1..7."""
    indicator_ids = exact_strategy["indicator_ids"]
    direction_mode_code_value = direction_mode_code(direction_mode)
    n_check = min(check_n, int(combo_chunk[indicator_ids[0]].shape[0]))
    if n_check <= 0:
        return {"checked": 0, "exact_backend": exact_strategy["name"], "max_abs_exact_backend_ret_diff": 0.0}

    subset = {indicator_id: np.ascontiguousarray(combo_chunk[indicator_id][:n_check]) for indicator_id in indicator_ids}
    backend_total_return_pct = np.empty(n_check, dtype=np.float64)
    backend_max_drawdown_pct = np.empty(n_check, dtype=np.float64)
    backend_return_over_max_drawdown = np.empty(n_check, dtype=np.float64)
    backend_profit_factor = np.empty(n_check, dtype=np.float64)
    backend_trade_count = np.empty(n_check, dtype=np.int32)
    backend_sharpe_trades = np.empty(n_check, dtype=np.float64)
    backend_win_rate_pct = np.empty(n_check, dtype=np.float64)
    backend_avg_trade_ret_pct = np.empty(n_check, dtype=np.float64)
    backend_avg_trade_exec_bars = np.empty(n_check, dtype=np.float64)
    backend_exposure_pct = np.empty(n_check, dtype=np.float64)

    evaluate_no_risk_exact_chunk(
        selected=subset,
        indicator_pools=indicator_pools,
        exact_strategy=exact_strategy,
        exact_context=exact_context,
        total_return_pct=backend_total_return_pct,
        max_drawdown_pct=backend_max_drawdown_pct,
        return_over_max_drawdown=backend_return_over_max_drawdown,
        profit_factor=backend_profit_factor,
        trade_count=backend_trade_count,
        sharpe_trades=backend_sharpe_trades,
        win_rate_pct=backend_win_rate_pct,
        avg_trade_ret_pct=backend_avg_trade_ret_pct,
        avg_trade_exec_bars=backend_avg_trade_exec_bars,
        exposure_pct=backend_exposure_pct,
        direction_mode_code_value=direction_mode_code_value,
    )

    reference_total_return_pct = np.empty(n_check, dtype=np.float64)
    reference_trade_count = np.empty(n_check, dtype=np.int32)
    for row_idx in range(n_check):
        local_indices = tuple(int(subset[indicator_id][row_idx]) for indicator_id in indicator_ids)
        metrics = evaluate_no_risk_reference_rows_slow(
            indicator_ids=indicator_ids,
            indicator_pools=indicator_pools,
            local_indices=local_indices,
            direction_mode=direction_mode,
        )
        reference_total_return_pct[row_idx] = float(metrics[0])
        reference_trade_count[row_idx] = int(metrics[4])

    if not np.array_equal(reference_trade_count, backend_trade_count):
        raise AssertionError(f"Exact backend {exact_strategy['name']!r} trade counts differ from generic slow reference.")
    max_abs_exact_backend_ret_diff = float(np.max(np.abs(reference_total_return_pct - backend_total_return_pct)))
    if max_abs_exact_backend_ret_diff > ret_tol:
        raise AssertionError(
            f"Exact backend {exact_strategy['name']!r} total return differs from generic slow reference by "
            f"{max_abs_exact_backend_ret_diff}, tolerance {ret_tol}."
        )
    return {
        "checked": n_check,
        "passed": True,
        "exact_backend": exact_strategy["name"],
        "direction_mode": direction_mode,
        "return_tolerance": ret_tol,
        "max_abs_exact_backend_ret_diff": max_abs_exact_backend_ret_diff,
        "trade_count_equal": True,
    }


def load_tp_sl_hit_times_15m(*, tp_values_pct, sl_values_pct):
    """Load and validate a TP/SL grid subset from published hit_times/15m artifacts."""
    t0 = time.perf_counter()
    hit_manifest = yaml.safe_load(HIT_TIMES_MANIFEST_15M_PATH.read_text())
    artifact_tp = np.load(TP_VALUES_15M_PATH, mmap_mode="r" if USE_MMAP else None)
    artifact_sl = np.load(SL_VALUES_15M_PATH, mmap_mode="r" if USE_MMAP else None)
    requested_tp = np.asarray(tp_values_pct, dtype=np.float32) / np.float32(100.0)
    requested_sl = np.asarray(sl_values_pct, dtype=np.float32) / np.float32(100.0)

    def grid_indices(artifact_values, requested_values, label):
        out = []
        for value in requested_values:
            matches = np.flatnonzero(np.isclose(artifact_values, value, rtol=0.0, atol=1e-7))
            if matches.size != 1:
                raise ValueError(f"Requested {label} level {float(value) * 100.0:.6g}% is not covered by hit_times/15m.")
            out.append(int(matches[0]))
        return np.asarray(out, dtype=np.int32)

    tp_idx = grid_indices(artifact_tp, requested_tp, "TP")
    sl_idx = grid_indices(artifact_sl, requested_sl, "SL")
    validation_time = time.perf_counter() - t0
    t1 = time.perf_counter()
    context = {
        "manifest": hit_manifest,
        "manifest_hash": hashlib.sha256(HIT_TIMES_MANIFEST_15M_PATH.read_bytes()).hexdigest(),
        "tp_values": np.ascontiguousarray(np.asarray(artifact_tp[tp_idx], dtype=np.float32)),
        "sl_values": np.ascontiguousarray(np.asarray(artifact_sl[sl_idx], dtype=np.float32)),
        "long_tp": np.ascontiguousarray(np.load(LONG_TP_15M_PATH, mmap_mode="r" if USE_MMAP else None)[tp_idx, :]),
        "long_sl": np.ascontiguousarray(np.load(LONG_SL_15M_PATH, mmap_mode="r" if USE_MMAP else None)[sl_idx, :]),
        "short_tp": np.ascontiguousarray(np.load(SHORT_TP_15M_PATH, mmap_mode="r" if USE_MMAP else None)[tp_idx, :]),
        "short_sl": np.ascontiguousarray(np.load(SHORT_SL_15M_PATH, mmap_mode="r" if USE_MMAP else None)[sl_idx, :]),
    }
    context["load_hit_times_s"] = time.perf_counter() - t1
    context["tp_sl_grid_validation_s"] = validation_time
    fee_two_sides = float((1.0 - FEE_RATE) * (1.0 - FEE_RATE))
    context["fee_two_sides"] = fee_two_sides
    context["log_fee_two_sides"] = float(math.log(fee_two_sides))
    context["long_tp_eq"] = np.ascontiguousarray((1.0 + context["tp_values"]).astype(np.float32))
    context["long_sl_eq"] = np.ascontiguousarray((1.0 - context["sl_values"]).astype(np.float32))
    context["short_tp_eq"] = np.ascontiguousarray((1.0 + context["tp_values"]).astype(np.float32))
    context["short_sl_eq"] = np.ascontiguousarray((1.0 - context["sl_values"]).astype(np.float32))
    context["log_fac_tp_long"] = np.ascontiguousarray(np.log(context["long_tp_eq"].astype(np.float64) * fee_two_sides))
    context["log_fac_sl_long"] = np.ascontiguousarray(np.log(context["long_sl_eq"].astype(np.float64) * fee_two_sides))
    context["log_fac_tp_short"] = np.ascontiguousarray(np.log(context["short_tp_eq"].astype(np.float64) * fee_two_sides))
    context["log_fac_sl_short"] = np.ascontiguousarray(np.log(context["short_sl_eq"].astype(np.float64) * fee_two_sides))
    return context


risk_price_open_15m = np.ascontiguousarray(price_fields_15m["open"])
risk_price_close_15m = np.ascontiguousarray(price_fields_15m["close"])
risk_log_open_15m = np.zeros(risk_price_open_15m.shape[0], dtype=np.float64)
risk_positive_open_15m = risk_price_open_15m > np.float32(0.0)
risk_log_open_15m[risk_positive_open_15m] = np.log(risk_price_open_15m[risk_positive_open_15m].astype(np.float64, copy=False))
risk_run_abs_start_15m = np.int32(time_slice_start_15m)
risk_T_exec_abs_15m = np.int32(time_slice_stop_15m)
risk_last_close_15m = float(risk_price_close_15m[int(risk_T_exec_abs_15m) - 1])
risk_log_last_close_15m = float(math.log(risk_last_close_15m)) if risk_last_close_15m > 0.0 else 0.0


@njit_cached(inline="always")
def tp_sl_add_row_range(row_diff: np.ndarray, row_i: np.int32, col_start: np.int32, col_stop: np.int32, value: float) -> None:
    if col_start < col_stop:
        row_diff[row_i, col_start] += value
        row_diff[row_i, col_stop] -= value


@njit_cached(inline="always")
def tp_sl_add_col_range(col_diff: np.ndarray, row_start: np.int32, row_stop: np.int32, col_i: np.int32, value: float) -> None:
    if row_start < row_stop:
        col_diff[row_start, col_i] += value
        col_diff[row_stop, col_i] -= value


@njit_cached(inline="always")
def tp_sl_add_rect(rect_diff: np.ndarray, row_start: np.int32, col_start: np.int32, row_stop: np.int32, col_stop: np.int32, value: float) -> None:
    if row_start < row_stop and col_start < col_stop:
        rect_diff[row_start, col_start] += value
        rect_diff[row_stop, col_start] -= value
        rect_diff[row_start, col_stop] -= value
        rect_diff[row_stop, col_stop] += value


@njit_cached(inline="always")
def tp_sl_lower_bound_ge_hit(hit_table: np.ndarray, start_exec: np.int32, n_levels: np.int32, target: np.int32) -> np.int32:
    lo = np.int32(0)
    hi = np.int32(n_levels)
    while lo < hi:
        mid = np.int32((lo + hi) // 2)
        if np.int32(hit_table[mid, start_exec]) >= target:
            hi = mid
        else:
            lo = np.int32(mid + 1)
    return lo


@njit_cached(inline="always")
def tp_sl_signal_exit_log_contrib(dirn: np.int8, entry_abs: np.int32, exit_abs: np.int32, price_open: np.ndarray, log_open: np.ndarray, log_fee: float) -> float:
    entry_open = float(price_open[entry_abs])
    exit_open = float(price_open[exit_abs])
    if entry_open <= 0.0 or exit_open <= 0.0:
        return NEG_LARGE
    entry_log = float(log_open[entry_abs])
    contrib = log_fee
    if dirn == 1:
        contrib += float(log_open[exit_abs]) - entry_log
    else:
        ratio = exit_open / entry_open
        if ratio >= 2.0:
            return NEG_LARGE
        contrib += math.log(2.0 - ratio)
    return contrib


@njit_cached(inline="always")
def tp_sl_final_close_log_contrib(dirn: np.int8, entry_abs: np.int32, price_open: np.ndarray, log_open: np.ndarray, last_close: float, log_last_close: float, log_fee: float) -> float:
    entry_open = float(price_open[entry_abs])
    if entry_open <= 0.0 or last_close <= 0.0:
        return NEG_LARGE
    entry_log = float(log_open[entry_abs])
    contrib = log_fee
    if dirn == 1:
        contrib += log_last_close - entry_log
    else:
        ratio = last_close / entry_open
        if ratio >= 2.0:
            return NEG_LARGE
        contrib += math.log(2.0 - ratio)
    return contrib


@njit_cached(inline="always")
def tp_sl_apply_trade_to_diff(
    dirn: np.int8,
    entry_abs: np.int32,
    sig_exit_abs: np.int32,
    price_open: np.ndarray,
    log_open: np.ndarray,
    last_close: float,
    log_last_close: float,
    T_exec_abs: np.int32,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    log_fac_tp_long: np.ndarray,
    log_fac_sl_long: np.ndarray,
    log_fac_tp_short: np.ndarray,
    log_fac_sl_short: np.ndarray,
    log_fee_two_sides: float,
    close_on_end: np.int8,
    row_diff: np.ndarray,
    col_diff: np.ndarray,
    rect_diff: np.ndarray,
) -> None:
    n_tp = np.int32(hit_long_tp.shape[0])
    n_sl = np.int32(hit_long_sl.shape[0])
    if float(price_open[entry_abs]) <= 0.0:
        return
    start = np.int32(entry_abs + 1)
    if dirn == 1:
        hit_tp = hit_long_tp
        hit_sl = hit_long_sl
        log_tp_arr = log_fac_tp_long
        log_sl_arr = log_fac_sl_long
    else:
        hit_tp = hit_short_tp
        hit_sl = hit_short_sl
        log_tp_arr = log_fac_tp_short
        log_sl_arr = log_fac_sl_short

    if start >= T_exec_abs:
        if sig_exit_abs < T_exec_abs:
            contrib = tp_sl_signal_exit_log_contrib(dirn, entry_abs, sig_exit_abs, price_open, log_open, log_fee_two_sides)
            tp_sl_add_rect(rect_diff, np.int32(0), np.int32(0), n_tp, n_sl, contrib)
        elif close_on_end == 1 and T_exec_abs > 0:
            contrib = tp_sl_final_close_log_contrib(dirn, entry_abs, price_open, log_open, last_close, log_last_close, log_fee_two_sides)
            tp_sl_add_rect(rect_diff, np.int32(0), np.int32(0), n_tp, n_sl, contrib)
        return

    if sig_exit_abs < T_exec_abs:
        i_sig = tp_sl_lower_bound_ge_hit(hit_tp, start, n_tp, sig_exit_abs)
        j_sig = tp_sl_lower_bound_ge_hit(hit_sl, start, n_sl, sig_exit_abs)
        contrib = tp_sl_signal_exit_log_contrib(dirn, entry_abs, sig_exit_abs, price_open, log_open, log_fee_two_sides)
        tp_sl_add_rect(rect_diff, i_sig, j_sig, n_tp, n_sl, contrib)

        j_ptr = np.int32(0)
        for i in range(i_sig):
            t_tp = np.int32(hit_tp[i, start])
            while j_ptr < j_sig and np.int32(hit_sl[j_ptr, start]) <= t_tp:
                j_ptr = np.int32(j_ptr + 1)
            tp_sl_add_row_range(row_diff, np.int32(i), j_ptr, n_sl, float(log_tp_arr[i]))

        i_ptr = np.int32(0)
        for j in range(j_sig):
            t_sl = np.int32(hit_sl[j, start])
            while i_ptr < i_sig and np.int32(hit_tp[i_ptr, start]) < t_sl:
                i_ptr = np.int32(i_ptr + 1)
            tp_sl_add_col_range(col_diff, i_ptr, n_tp, np.int32(j), float(log_sl_arr[j]))
    else:
        i_end = tp_sl_lower_bound_ge_hit(hit_tp, start, n_tp, T_exec_abs)
        j_end = tp_sl_lower_bound_ge_hit(hit_sl, start, n_sl, T_exec_abs)
        if close_on_end == 1 and T_exec_abs > 0:
            contrib = tp_sl_final_close_log_contrib(dirn, entry_abs, price_open, log_open, last_close, log_last_close, log_fee_two_sides)
            tp_sl_add_rect(rect_diff, i_end, j_end, n_tp, n_sl, contrib)

        j_ptr = np.int32(0)
        for i in range(i_end):
            t_tp = np.int32(hit_tp[i, start])
            while j_ptr < n_sl and np.int32(hit_sl[j_ptr, start]) <= t_tp:
                j_ptr = np.int32(j_ptr + 1)
            tp_sl_add_row_range(row_diff, np.int32(i), j_ptr, n_sl, float(log_tp_arr[i]))

        i_ptr = np.int32(0)
        for j in range(j_end):
            t_sl = np.int32(hit_sl[j, start])
            while i_ptr < i_end and np.int32(hit_tp[i_ptr, start]) < t_sl:
                i_ptr = np.int32(i_ptr + 1)
            tp_sl_add_col_range(col_diff, i_ptr, n_tp, np.int32(j), float(log_sl_arr[j]))


@njit_cached(parallel=True, fastmath=True)
def evaluate_best_tp_sl_event_segments_n(
    combo_idx_by_indicator: np.ndarray,
    segment_starts: np.ndarray,
    segment_ends: np.ndarray,
    segment_values: np.ndarray,
    segment_counts: np.ndarray,
    segment_pos_workspace: np.ndarray,
    run_abs_start: np.int32,
    T_exec_abs: np.int32,
    price_open: np.ndarray,
    log_open: np.ndarray,
    last_close: float,
    log_last_close: float,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    log_fac_tp_long: np.ndarray,
    log_fac_sl_long: np.ndarray,
    log_fac_tp_short: np.ndarray,
    log_fac_sl_short: np.ndarray,
    log_fee_two_sides: float,
    close_on_end: np.int8,
    direction_mode: np.int8,
    out_best_tp_idx: np.ndarray,
    out_best_sl_idx: np.ndarray,
    out_best_ret: np.ndarray,
    out_trade_count: np.ndarray,
) -> None:
    arity = combo_idx_by_indicator.shape[0]
    K = combo_idx_by_indicator.shape[1]
    n_tp = hit_long_tp.shape[0]
    n_sl = hit_long_sl.shape[0]

    for k in nb.prange(K):
        for indicator_pos in range(arity):
            segment_pos_workspace[k, indicator_pos] = np.int32(0)
        row_diff = np.zeros((n_tp, n_sl + 1), dtype=np.float64)
        col_diff = np.zeros((n_tp + 1, n_sl), dtype=np.float64)
        rect_diff = np.zeros((n_tp + 1, n_sl + 1), dtype=np.float64)
        current_dir = np.int8(0)
        current_entry_abs = np.int32(0)
        trade_count = np.int32(0)

        while True:
            active = True
            segment_start = np.int32(0)
            segment_end = np.int32(2147483647)
            for indicator_pos in range(arity):
                row_idx = combo_idx_by_indicator[indicator_pos, k]
                segment_idx = segment_pos_workspace[k, indicator_pos]
                if segment_idx >= segment_counts[indicator_pos, row_idx]:
                    active = False
                    break
                start_value = segment_starts[indicator_pos, row_idx, segment_idx]
                end_value = segment_ends[indicator_pos, row_idx, segment_idx]
                if start_value > segment_start:
                    segment_start = start_value
                if end_value < segment_end:
                    segment_end = end_value
            if not active:
                break

            if segment_start < segment_end:
                first_row_idx = combo_idx_by_indicator[0, k]
                first_segment_idx = segment_pos_workspace[k, 0]
                raw_dir = segment_values[0, first_row_idx, first_segment_idx]
                if raw_dir != 0:
                    for indicator_pos in range(1, arity):
                        row_idx = combo_idx_by_indicator[indicator_pos, k]
                        segment_idx = segment_pos_workspace[k, indicator_pos]
                        if segment_values[indicator_pos, row_idx, segment_idx] != raw_dir:
                            raw_dir = np.int8(0)
                            break
                dirn = apply_direction_mode(raw_dir, direction_mode)
                if dirn != 0 or (direction_mode == 1 and current_dir != 0):
                    entry_abs = np.int32(run_abs_start + segment_start + 1)
                    if entry_abs >= T_exec_abs:
                        break
                    if dirn == 0:
                        tp_sl_apply_trade_to_diff(
                            current_dir,
                            current_entry_abs,
                            entry_abs,
                            price_open,
                            log_open,
                            last_close,
                            log_last_close,
                            T_exec_abs,
                            hit_long_tp,
                            hit_long_sl,
                            hit_short_tp,
                            hit_short_sl,
                            log_fac_tp_long,
                            log_fac_sl_long,
                            log_fac_tp_short,
                            log_fac_sl_short,
                            log_fee_two_sides,
                            close_on_end,
                            row_diff,
                            col_diff,
                            rect_diff,
                        )
                        trade_count += np.int32(1)
                        current_dir = np.int8(0)
                        current_entry_abs = np.int32(0)
                    elif current_dir == 0:
                        current_dir = dirn
                        current_entry_abs = entry_abs
                    elif dirn != current_dir:
                        tp_sl_apply_trade_to_diff(
                            current_dir,
                            current_entry_abs,
                            entry_abs,
                            price_open,
                            log_open,
                            last_close,
                            log_last_close,
                            T_exec_abs,
                            hit_long_tp,
                            hit_long_sl,
                            hit_short_tp,
                            hit_short_sl,
                            log_fac_tp_long,
                            log_fac_sl_long,
                            log_fac_tp_short,
                            log_fac_sl_short,
                            log_fee_two_sides,
                            close_on_end,
                            row_diff,
                            col_diff,
                            rect_diff,
                        )
                        trade_count += np.int32(1)
                        current_dir = dirn
                        current_entry_abs = entry_abs

            for indicator_pos in range(arity):
                row_idx = combo_idx_by_indicator[indicator_pos, k]
                segment_idx = segment_pos_workspace[k, indicator_pos]
                if segment_ends[indicator_pos, row_idx, segment_idx] == segment_end:
                    segment_pos_workspace[k, indicator_pos] = np.int32(segment_idx + 1)

        if current_dir != 0 and close_on_end == 1 and T_exec_abs > 0:
            tp_sl_apply_trade_to_diff(
                current_dir,
                current_entry_abs,
                T_exec_abs,
                price_open,
                log_open,
                last_close,
                log_last_close,
                T_exec_abs,
                hit_long_tp,
                hit_long_sl,
                hit_short_tp,
                hit_short_sl,
                log_fac_tp_long,
                log_fac_sl_long,
                log_fac_tp_short,
                log_fac_sl_short,
                log_fee_two_sides,
                close_on_end,
                row_diff,
                col_diff,
                rect_diff,
            )
            trade_count += np.int32(1)

        for i in range(n_tp):
            run = 0.0
            for j in range(n_sl):
                run += row_diff[i, j]
                row_diff[i, j] = run
        for j in range(n_sl):
            run = 0.0
            for i in range(n_tp):
                run += col_diff[i, j]
                col_diff[i, j] = run
        for i in range(n_tp):
            row_run = 0.0
            for j in range(n_sl):
                row_run += rect_diff[i, j]
                if i == 0:
                    rect_diff[i, j] = row_run
                else:
                    rect_diff[i, j] = row_run + rect_diff[i - 1, j]

        best_log = -1.0e300
        best_tp = np.int32(0)
        best_sl = np.int32(0)
        for i in range(n_tp):
            for j in range(n_sl):
                value = row_diff[i, j] + col_diff[i, j] + rect_diff[i, j]
                if value > best_log:
                    best_log = value
                    best_tp = np.int32(i)
                    best_sl = np.int32(j)
        out_best_tp_idx[k] = best_tp
        out_best_sl_idx[k] = best_sl
        out_trade_count[k] = trade_count
        if trade_count <= 0:
            out_best_ret[k] = np.float32(0.0)
        elif best_log <= -1.0e200:
            out_best_ret[k] = np.float32(-1.0)
        else:
            out_best_ret[k] = np.float32(math.exp(best_log) - 1.0)


def build_trade_list_15m_for_indicator_rows_slow(*, indicator_ids: tuple[str, ...], indicator_pools: dict, local_indices: tuple[int, ...], direction_mode: str = DIRECTION_MODE_LONG_SHORT_REVERSAL, close_on_end: np.int8 = CLOSE_ON_END):
    """Build a 15m-indexed trade list for TP/SL hit-time reference scoring."""
    rows = [indicator_pools[indicator_id]["trade_T"][local_indices[pos]] for pos, indicator_id in enumerate(indicator_ids)]
    n_sig = rows[0].shape[0]
    entry_abs = []
    directions = []
    sig_exit_abs = []
    current_dir = 0
    current_entry = 0
    for t in range(n_sig):
        raw_dir = int(rows[0][t])
        if raw_dir != 0:
            for row in rows[1:]:
                if int(row[t]) != raw_dir:
                    raw_dir = 0
                    break
        dirn = apply_direction_mode_py(raw_dir, direction_mode)
        if dirn == 0 and not (direction_mode == DIRECTION_MODE_LONG_ONLY and current_dir != 0):
            continue
        entry_idx = int(time_slice_start_15m + t + 1)
        if entry_idx >= int(risk_T_exec_abs_15m):
            break
        if dirn == 0:
            entry_abs.append(current_entry)
            directions.append(current_dir)
            sig_exit_abs.append(entry_idx)
            current_dir = 0
            current_entry = 0
            continue
        if current_dir == 0:
            current_dir = dirn
            current_entry = entry_idx
            continue
        if dirn == current_dir:
            continue
        entry_abs.append(current_entry)
        directions.append(current_dir)
        sig_exit_abs.append(entry_idx)
        current_dir = dirn
        current_entry = entry_idx
    if current_dir != 0 and close_on_end == 1:
        entry_abs.append(current_entry)
        directions.append(current_dir)
        sig_exit_abs.append(int(risk_T_exec_abs_15m))
    return (
        np.asarray(entry_abs, dtype=np.int32),
        np.asarray(directions, dtype=np.int8),
        np.asarray(sig_exit_abs, dtype=np.int32),
    )



def tp_sl_signal_exit_log_contrib_reference(dirn: int, entry_abs: int, exit_abs: int, *, risk_context: dict) -> float:
    """Python reference equivalent of TP/SL signal-exit log contribution."""
    entry_open = float(risk_price_open_15m[entry_abs])
    exit_open = float(risk_price_open_15m[exit_abs])
    if entry_open <= 0.0 or exit_open <= 0.0:
        return NEG_LARGE
    if dirn == 1:
        return float(risk_context["log_fee_two_sides"]) + math.log(exit_open / entry_open)
    ratio = exit_open / entry_open
    if ratio >= 2.0:
        return NEG_LARGE
    return float(risk_context["log_fee_two_sides"]) + math.log(2.0 - ratio)


def tp_sl_final_close_log_contrib_reference(dirn: int, entry_abs: int, *, risk_context: dict) -> float:
    """Python reference equivalent of TP/SL final-close log contribution."""
    entry_open = float(risk_price_open_15m[entry_abs])
    if entry_open <= 0.0 or risk_last_close_15m <= 0.0:
        return NEG_LARGE
    if dirn == 1:
        return float(risk_context["log_fee_two_sides"]) + math.log(risk_last_close_15m / entry_open)
    ratio = risk_last_close_15m / entry_open
    if ratio >= 2.0:
        return NEG_LARGE
    return float(risk_context["log_fee_two_sides"]) + math.log(2.0 - ratio)


def tp_sl_trade_log_contrib_reference(dirn: int, entry_abs: int, sig_exit_abs: int, tp_i: int, sl_i: int, *, risk_context: dict) -> float:
    """Reference one-trade TP/SL contribution for one TP/SL cell."""
    if float(risk_price_open_15m[entry_abs]) <= 0.0:
        return 0.0
    start = int(entry_abs + 1)
    T_exec_abs = int(risk_T_exec_abs_15m)
    if dirn == 1:
        hit_tp = risk_context["long_tp"]
        hit_sl = risk_context["long_sl"]
        log_tp = risk_context["log_fac_tp_long"]
        log_sl = risk_context["log_fac_sl_long"]
    else:
        hit_tp = risk_context["short_tp"]
        hit_sl = risk_context["short_sl"]
        log_tp = risk_context["log_fac_tp_short"]
        log_sl = risk_context["log_fac_sl_short"]

    if start < T_exec_abs:
        t_tp = int(hit_tp[tp_i, start])
        t_sl = int(hit_sl[sl_i, start])
        stop_abs = int(sig_exit_abs) if int(sig_exit_abs) < T_exec_abs else T_exec_abs
        if t_tp < stop_abs and t_tp < t_sl:
            return float(log_tp[tp_i])
        if t_sl < stop_abs and t_sl <= t_tp:
            return float(log_sl[sl_i])

    if int(sig_exit_abs) < T_exec_abs:
        return tp_sl_signal_exit_log_contrib_reference(dirn, entry_abs, int(sig_exit_abs), risk_context=risk_context)
    if CLOSE_ON_END == 1 and T_exec_abs > 0:
        return tp_sl_final_close_log_contrib_reference(dirn, entry_abs, risk_context=risk_context)
    return 0.0


@njit_cached(inline="always")
def tp_sl_trade_log_contrib_reference_kernel(
    dirn: np.int8,
    entry_abs: np.int32,
    sig_exit_abs: np.int32,
    tp_i: np.int32,
    sl_i: np.int32,
    price_open: np.ndarray,
    last_close: float,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    log_fac_tp_long: np.ndarray,
    log_fac_sl_long: np.ndarray,
    log_fac_tp_short: np.ndarray,
    log_fac_sl_short: np.ndarray,
    log_fee_two_sides: float,
    close_on_end: np.int8,
    T_exec_abs: np.int32,
) -> float:
    entry_open = float(price_open[entry_abs])
    if entry_open <= 0.0:
        return 0.0
    start = np.int32(entry_abs + 1)
    if dirn == 1:
        hit_tp_value = np.int32(hit_long_tp[tp_i, start]) if start < T_exec_abs else np.int32(2147483647)
        hit_sl_value = np.int32(hit_long_sl[sl_i, start]) if start < T_exec_abs else np.int32(2147483647)
        log_tp = float(log_fac_tp_long[tp_i])
        log_sl = float(log_fac_sl_long[sl_i])
    else:
        hit_tp_value = np.int32(hit_short_tp[tp_i, start]) if start < T_exec_abs else np.int32(2147483647)
        hit_sl_value = np.int32(hit_short_sl[sl_i, start]) if start < T_exec_abs else np.int32(2147483647)
        log_tp = float(log_fac_tp_short[tp_i])
        log_sl = float(log_fac_sl_short[sl_i])

    stop_abs = sig_exit_abs if sig_exit_abs < T_exec_abs else T_exec_abs
    if start < T_exec_abs:
        if hit_tp_value < stop_abs and hit_tp_value < hit_sl_value:
            return log_tp
        if hit_sl_value < stop_abs and hit_sl_value <= hit_tp_value:
            return log_sl

    if sig_exit_abs < T_exec_abs:
        exit_open = float(price_open[sig_exit_abs])
        if exit_open <= 0.0:
            return NEG_LARGE
        if dirn == 1:
            return log_fee_two_sides + math.log(exit_open / entry_open)
        ratio = exit_open / entry_open
        if ratio >= 2.0:
            return NEG_LARGE
        return log_fee_two_sides + math.log(2.0 - ratio)

    if close_on_end == 1 and T_exec_abs > 0:
        if last_close <= 0.0:
            return NEG_LARGE
        if dirn == 1:
            return log_fee_two_sides + math.log(last_close / entry_open)
        ratio = last_close / entry_open
        if ratio >= 2.0:
            return NEG_LARGE
        return log_fee_two_sides + math.log(2.0 - ratio)
    return 0.0


@njit_cached(fastmath=False)
def evaluate_tp_sl_reference_trade_list_direct(
    entry_abs: np.ndarray,
    dir_arr: np.ndarray,
    sig_exit_abs: np.ndarray,
    price_open: np.ndarray,
    last_close: float,
    hit_long_tp: np.ndarray,
    hit_long_sl: np.ndarray,
    hit_short_tp: np.ndarray,
    hit_short_sl: np.ndarray,
    log_fac_tp_long: np.ndarray,
    log_fac_sl_long: np.ndarray,
    log_fac_tp_short: np.ndarray,
    log_fac_sl_short: np.ndarray,
    log_fee_two_sides: float,
    close_on_end: np.int8,
    T_exec_abs: np.int32,
) -> tuple[np.int32, np.int32, float, np.int32]:
    n_trades = np.int32(entry_abs.shape[0])
    if n_trades == 0:
        return np.int32(0), np.int32(0), 0.0, np.int32(0)
    n_tp = np.int32(hit_long_tp.shape[0])
    n_sl = np.int32(hit_long_sl.shape[0])
    best_log = -1.0e300
    best_tp = np.int32(0)
    best_sl = np.int32(0)
    for tp_i in range(n_tp):
        for sl_i in range(n_sl):
            total_log = 0.0
            for trade_idx in range(n_trades):
                total_log += tp_sl_trade_log_contrib_reference_kernel(
                    dir_arr[trade_idx],
                    entry_abs[trade_idx],
                    sig_exit_abs[trade_idx],
                    np.int32(tp_i),
                    np.int32(sl_i),
                    price_open,
                    last_close,
                    hit_long_tp,
                    hit_long_sl,
                    hit_short_tp,
                    hit_short_sl,
                    log_fac_tp_long,
                    log_fac_sl_long,
                    log_fac_tp_short,
                    log_fac_sl_short,
                    log_fee_two_sides,
                    close_on_end,
                    T_exec_abs,
                )
            if total_log > best_log:
                best_log = total_log
                best_tp = np.int32(tp_i)
                best_sl = np.int32(sl_i)
    if best_log <= -1.0e200:
        best_ret = -1.0
    else:
        best_ret = math.exp(best_log) - 1.0
    return best_tp, best_sl, best_ret, n_trades


def evaluate_tp_sl_reference_rows_slow(*, indicator_ids: tuple[str, ...], indicator_pools: dict, local_indices: tuple[int, ...], risk_context: dict, direction_mode: str = DIRECTION_MODE_LONG_SHORT_REVERSAL):
    """Direct slow-path TP/SL reference for one combo and the requested hit_times/15m grid."""
    entry_abs, dir_arr, sig_exit_abs = build_trade_list_15m_for_indicator_rows_slow(
        indicator_ids=indicator_ids,
        indicator_pools=indicator_pools,
        local_indices=local_indices,
        direction_mode=direction_mode,
    )
    ref_tp, ref_sl, ref_ret, ref_trades = evaluate_tp_sl_reference_trade_list_direct(
        entry_abs,
        dir_arr,
        sig_exit_abs,
        risk_price_open_15m,
        risk_last_close_15m,
        risk_context["long_tp"],
        risk_context["long_sl"],
        risk_context["short_tp"],
        risk_context["short_sl"],
        risk_context["log_fac_tp_long"],
        risk_context["log_fac_sl_long"],
        risk_context["log_fac_tp_short"],
        risk_context["log_fac_sl_short"],
        risk_context["log_fee_two_sides"],
        CLOSE_ON_END,
        risk_T_exec_abs_15m,
    )
    return int(ref_tp), int(ref_sl), float(ref_ret), int(ref_trades)



def run_tp_sl_self_check(*, combo_chunk: dict, indicator_pools: dict, indicator_ids: tuple[str, ...], risk_context: dict, check_n: int = SELF_CHECK_N_DEFAULT, ret_tol: float = 5e-5, direction_mode: str = DIRECTION_MODE_LONG_SHORT_REVERSAL):
    """Bounded TP/SL self-check for arity 1..7 against a slow Python reference."""
    direction_mode_code_value = direction_mode_code(direction_mode)
    n_check = min(check_n, int(combo_chunk[indicator_ids[0]].shape[0]))
    if n_check <= 0:
        return {"checked": 0, "passed": True}
    selected = {indicator_id: np.ascontiguousarray(combo_chunk[indicator_id][:n_check]) for indicator_id in indicator_ids}
    exact_context = build_segment_stack(indicator_ids=indicator_ids, indicator_pools=indicator_pools)
    combo_idx_by_indicator = make_combo_idx_matrix(combo_chunk=selected, indicator_ids=indicator_ids)
    segment_pos_workspace = np.empty((combo_idx_by_indicator.shape[1], len(indicator_ids)), dtype=np.int32)
    best_tp_idx = np.empty(n_check, dtype=np.int32)
    best_sl_idx = np.empty(n_check, dtype=np.int32)
    best_ret = np.empty(n_check, dtype=np.float32)
    trade_count = np.empty(n_check, dtype=np.int32)
    evaluate_best_tp_sl_event_segments_n(
        combo_idx_by_indicator,
        exact_context["starts"],
        exact_context["ends"],
        exact_context["values"],
        exact_context["counts"],
        segment_pos_workspace,
        risk_run_abs_start_15m,
        risk_T_exec_abs_15m,
        risk_price_open_15m,
        risk_log_open_15m,
        risk_last_close_15m,
        risk_log_last_close_15m,
        risk_context["long_tp"],
        risk_context["long_sl"],
        risk_context["short_tp"],
        risk_context["short_sl"],
        risk_context["log_fac_tp_long"],
        risk_context["log_fac_sl_long"],
        risk_context["log_fac_tp_short"],
        risk_context["log_fac_sl_short"],
        risk_context["log_fee_two_sides"],
        CLOSE_ON_END,
        direction_mode_code_value,
        best_tp_idx,
        best_sl_idx,
        best_ret,
        trade_count,
    )
    if np.any(best_tp_idx < 0) or np.any(best_tp_idx >= risk_context["tp_values"].shape[0]):
        raise AssertionError("TP/SL backend produced an out-of-range TP index.")
    if np.any(best_sl_idx < 0) or np.any(best_sl_idx >= risk_context["sl_values"].shape[0]):
        raise AssertionError("TP/SL backend produced an out-of-range SL index.")
    if np.any(~np.isfinite(best_ret)):
        raise AssertionError("TP/SL backend produced non-finite returns.")
    if np.any(trade_count < 0):
        raise AssertionError("TP/SL backend produced negative trade counts.")

    reference_best_tp_idx = np.empty(n_check, dtype=np.int32)
    reference_best_sl_idx = np.empty(n_check, dtype=np.int32)
    reference_best_ret = np.empty(n_check, dtype=np.float64)
    reference_trade_count = np.empty(n_check, dtype=np.int32)
    for row_idx in range(n_check):
        local_indices = tuple(int(selected[indicator_id][row_idx]) for indicator_id in indicator_ids)
        ref_tp, ref_sl, ref_ret, ref_trades = evaluate_tp_sl_reference_rows_slow(
            indicator_ids=indicator_ids,
            indicator_pools=indicator_pools,
            local_indices=local_indices,
            risk_context=risk_context,
            direction_mode=direction_mode,
        )
        reference_best_tp_idx[row_idx] = ref_tp
        reference_best_sl_idx[row_idx] = ref_sl
        reference_best_ret[row_idx] = ref_ret
        reference_trade_count[row_idx] = ref_trades

    if not np.array_equal(reference_trade_count, trade_count):
        raise AssertionError("TP/SL backend trade counts differ from slow reference.")
    ret_diff = np.abs(reference_best_ret - best_ret.astype(np.float64))
    max_abs_ret_diff = float(np.max(ret_diff)) if ret_diff.size else 0.0
    if max_abs_ret_diff > ret_tol:
        raise AssertionError(f"TP/SL backend best return differs from slow reference by {max_abs_ret_diff}, tolerance {ret_tol}.")
    exact_best_cell_equal = bool(np.array_equal(reference_best_tp_idx, best_tp_idx) and np.array_equal(reference_best_sl_idx, best_sl_idx))
    if not exact_best_cell_equal:
        # A different cell is acceptable only when it is return-equivalent inside the same tolerance.
        if max_abs_ret_diff > ret_tol:
            raise AssertionError("TP/SL backend best TP/SL cell differs from slow reference with a material return difference.")
    return {
        "checked": n_check,
        "passed": True,
        "direction_mode": direction_mode,
        "return_tolerance": ret_tol,
        "max_abs_tp_sl_ret_diff": max_abs_ret_diff,
        "trade_count_equal": True,
        "best_cell_equal": exact_best_cell_equal,
        "valid_tp_sl_indexes": True,
    }


def current_process_metrics_start():
    try:
        import psutil
        process = psutil.Process()
        process.cpu_percent(None)
        return {
            "process": process,
            "cpu_start": process.cpu_times(),
            "rss_start": process.memory_info().rss,
            "thread_count_start": process.num_threads(),
            "samples": [],
            "rss_samples": [],
        }
    except Exception:
        return {"process": None, "samples": [], "rss_samples": []}


def current_process_metrics_stop(metrics, wall_time_s: float) -> dict:
    process = metrics.get("process")
    ru_maxrss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    peak_rss_bytes = int(ru_maxrss if platform.system() == "Darwin" else ru_maxrss * 1024)
    if process is None:
        return {
            "wall_time_s": wall_time_s,
            "process_cpu_time_s": None,
            "process_cpu_percent_equivalent": None,
            "peak_rss_mb": peak_rss_bytes / (1024 * 1024),
            "rss_delta_mb": None,
            "thread_count": None,
            "numba_threads": nb.get_num_threads(),
        }
    cpu_stop = process.cpu_times()
    rss_stop = process.memory_info().rss
    cpu_start = metrics["cpu_start"]
    process_cpu_time_s = (cpu_stop.user + cpu_stop.system) - (cpu_start.user + cpu_start.system)
    rss_start = metrics["rss_start"]
    return {
        "wall_time_s": wall_time_s,
        "process_cpu_time_s": process_cpu_time_s,
        "process_cpu_percent_equivalent": (process_cpu_time_s / wall_time_s * 100.0) if wall_time_s > 0.0 else 0.0,
        "peak_rss_mb": peak_rss_bytes / (1024 * 1024),
        "rss_delta_mb": (rss_stop - rss_start) / (1024 * 1024),
        "thread_count": process.num_threads(),
        "thread_count_start": metrics["thread_count_start"],
        "numba_threads": nb.get_num_threads(),
    }


def result_hash(top_results) -> str:
    return canonical_json_hash(top_results)


def top_rows_for_request(*, indicator_ids: tuple[str, ...], rows_per_indicator: int) -> dict:
    return {
        indicator_id: row_ids_for_sources(indicator_id=indicator_id, source_names=["close"])[:rows_per_indicator]
        for indicator_id in indicator_ids
    }


def run_no_risk_benchmark_once(*, indicator_ids: tuple[str, ...], rows_per_indicator: int, top_k: int, self_check_n: int, direction_mode: str) -> dict:
    row_pools = top_rows_for_request(indicator_ids=indicator_ids, rows_per_indicator=rows_per_indicator)
    metrics = current_process_metrics_start()
    t0 = time.perf_counter()
    result = search_topk_indicator_no_risk(
        indicator_ids=indicator_ids,
        row_pools=row_pools,
        indicator_top_frac=1.0,
        min_nonzero=PREFILTER_MIN_NONZERO,
        combo_top_frac=1.0,
        combo_min_confirm=1,
        combo_chunk_size=COMBO_CHUNK_SIZE,
        top_k=top_k,
        self_check_n=self_check_n,
        exact_backend=EXACT_BACKEND_AUTO,
        max_cartesian_candidates=None,
        direction_mode=direction_mode,
        verbose=False,
    )
    wall = time.perf_counter() - t0
    result["direction_mode"] = direction_mode
    result["runtime_metrics"] = current_process_metrics_stop(metrics, wall)
    result["result_hash"] = result_hash(result["top_results"])
    return result


def run_tp_sl_benchmark_once(*, indicator_ids: tuple[str, ...], rows_per_indicator: int, top_k: int, self_check_n: int, risk_context: dict, direction_mode: str) -> dict:
    direction_mode_code_value = direction_mode_code(direction_mode)
    total_start = time.perf_counter()
    timers = {
        "service_warmup": 0.0,
        "numba_warmup": 0.0,
        "prepare_pools": 0.0,
        "build_exact_context": 0.0,
        "build_proxy_context": 0.0,
        "combo_iteration": 0.0,
        "proxy_filter": 0.0,
        "self_check": 0.0,
        "exact_scoring": 0.0,
        "heap_update": 0.0,
        "top_result_proxy_fill": 0.0,
        "load_hit_times": risk_context.get("load_hit_times_s", 0.0),
        "tp_sl_grid_validation": risk_context.get("tp_sl_grid_validation_s", 0.0),
        "tp_sl_exact_scoring": 0.0,
        "total_without_warmup": 0.0,
    }
    row_pools = top_rows_for_request(indicator_ids=indicator_ids, rows_per_indicator=rows_per_indicator)
    metrics = current_process_metrics_start()

    t0 = time.perf_counter()
    indicator_pools = prepare_indicator_pools(
        indicator_ids=indicator_ids,
        row_pools=row_pools,
        top_frac=1.0,
        min_nonzero=PREFILTER_MIN_NONZERO,
        fee_rate=FEE_RATE,
        time_chunk=TIME_CHUNK,
    )
    timers["prepare_pools"] += time.perf_counter() - t0
    t0 = time.perf_counter()
    exact_context = build_segment_stack(indicator_ids=indicator_ids, indicator_pools=indicator_pools)
    timers["build_exact_context"] += time.perf_counter() - t0

    local_row_pools = {
        indicator_id: np.arange(indicator_pools[indicator_id]["trade_T"].shape[0], dtype=np.int32)
        for indicator_id in indicator_ids
    }
    heap = []
    total_combo_chunks = 0
    total_exact_candidates = 0
    self_check = None
    combo_iter = iter_combo_chunks(indicator_ids=indicator_ids, local_row_pools=local_row_pools, chunk_size=COMBO_CHUNK_SIZE)
    while True:
        t0 = time.perf_counter()
        try:
            combo_chunk = next(combo_iter)
        except StopIteration:
            timers["combo_iteration"] += time.perf_counter() - t0
            break
        timers["combo_iteration"] += time.perf_counter() - t0
        total_combo_chunks += 1
        chunk_len = int(combo_chunk[indicator_ids[0]].shape[0])
        total_exact_candidates += chunk_len
        if self_check is None and self_check_n > 0:
            t0 = time.perf_counter()
            self_check = run_tp_sl_self_check(
                combo_chunk=combo_chunk,
                indicator_pools=indicator_pools,
                indicator_ids=indicator_ids,
                risk_context=risk_context,
                check_n=self_check_n,
                direction_mode=direction_mode,
            )
            timers["self_check"] += time.perf_counter() - t0

        combo_idx_by_indicator = make_combo_idx_matrix(combo_chunk=combo_chunk, indicator_ids=indicator_ids)
        segment_pos_workspace = np.empty((combo_idx_by_indicator.shape[1], len(indicator_ids)), dtype=np.int32)
        best_tp_idx = np.empty(chunk_len, dtype=np.int32)
        best_sl_idx = np.empty(chunk_len, dtype=np.int32)
        best_ret = np.empty(chunk_len, dtype=np.float32)
        trade_count = np.empty(chunk_len, dtype=np.int32)
        t0 = time.perf_counter()
        evaluate_best_tp_sl_event_segments_n(
            combo_idx_by_indicator,
            exact_context["starts"],
            exact_context["ends"],
            exact_context["values"],
            exact_context["counts"],
            segment_pos_workspace,
            risk_run_abs_start_15m,
            risk_T_exec_abs_15m,
            risk_price_open_15m,
            risk_log_open_15m,
            risk_last_close_15m,
            risk_log_last_close_15m,
            risk_context["long_tp"],
            risk_context["long_sl"],
            risk_context["short_tp"],
            risk_context["short_sl"],
            risk_context["log_fac_tp_long"],
            risk_context["log_fac_sl_long"],
            risk_context["log_fac_tp_short"],
            risk_context["log_fac_sl_short"],
            risk_context["log_fee_two_sides"],
            CLOSE_ON_END,
            direction_mode_code_value,
            best_tp_idx,
            best_sl_idx,
            best_ret,
            trade_count,
        )
        elapsed = time.perf_counter() - t0
        timers["exact_scoring"] += elapsed
        timers["tp_sl_exact_scoring"] += elapsed
        t0 = time.perf_counter()
        for local_idx in range(chunk_len):
            score_pct = float(best_ret[local_idx]) * 100.0
            item = {
                "total_return_pct": score_pct,
                "best_tp_pct": float(risk_context["tp_values"][int(best_tp_idx[local_idx])] * 100.0),
                "best_sl_pct": float(risk_context["sl_values"][int(best_sl_idx[local_idx])] * 100.0),
                "trade_count": int(trade_count[local_idx]),
            }
            candidate_ordinal = int(total_exact_candidates - chunk_len + local_idx)
            heap_key = (score_pct, item["best_tp_pct"], item["best_sl_pct"], -candidate_ordinal)
            heap_item = (heap_key, item)
            if len(heap) < top_k:
                heapq.heappush(heap, heap_item)
            elif heap_key > heap[0][0]:
                heapq.heapreplace(heap, heap_item)
        timers["heap_update"] += time.perf_counter() - t0

    top_results = [item for _, item in sorted(heap, key=lambda pair: pair[0], reverse=True)]
    timers["total_without_warmup"] = time.perf_counter() - total_start
    wall = timers["total_without_warmup"]
    return {
        "indicator_ids": indicator_ids,
        "risk_mode": "tp_sl_grid",
        "direction_mode": direction_mode,
        "exact_engine": f"event_segments_{len(indicator_ids)}_tp_sl_15m_grid",
        "filtered_pool_sizes": {indicator_id: int(indicator_pools[indicator_id]["trade_T"].shape[0]) for indicator_id in indicator_ids},
        "cartesian_combinations": int(math.prod([indicator_pools[indicator_id]["trade_T"].shape[0] for indicator_id in indicator_ids])),
        "combo_chunks_processed": total_combo_chunks,
        "exact_candidates_evaluated": total_exact_candidates,
        "tp_levels": int(risk_context["tp_values"].shape[0]),
        "sl_levels": int(risk_context["sl_values"].shape[0]),
        "risk_cells_per_combo": int(risk_context["tp_values"].shape[0] * risk_context["sl_values"].shape[0]),
        "self_check": self_check,
        "timers": timers,
        "runtime_metrics": current_process_metrics_stop(metrics, wall),
        "top_results": top_results,
        "result_hash": result_hash(top_results),
    }


def benchmark_methodology_payload(*, rows_per_indicator: int, warmup_rows_per_indicator: int) -> dict:
    """Return the benchmark methodology persisted into JSON and Markdown evidence."""
    return {
        "host": "macstudio",
        "period": BACKTEST_REQUEST["period"],
        "workload": {
            "rows_per_indicator": rows_per_indicator,
            "warmup_rows_per_indicator": warmup_rows_per_indicator,
            "arities": list(INDICATOR_ARITIES),
            "risk_modes": ["none", "tp_sl_grid"],
            "direction_modes": list(DIRECTION_MODES),
            "tp_sl_grid": {"start_pct": 2.0, "stop_pct": 25.0, "step_pct": 0.5, "cells": 2209},
        },
        "indicator_sets": {str(arity): list(indicator_ids) for arity, indicator_ids in INDICATOR_ARITY_SETS.items()},
        "direction_semantics": {
            DIRECTION_MODE_LONG_ONLY: "+1 opens/holds long; 0 or -1 closes an open long; short trades are never opened.",
            DIRECTION_MODE_LONG_SHORT_REVERSAL: "+1 opens/holds long; -1 opens/holds short; opposite direction closes and reverses.",
        },
        "warmup": "Each measured run is preceded by a sample warmup on min(2, rows_per_indicator) rows per indicator for the same arity, risk mode, direction mode, and backend. Full dry-run warmup is intentionally not used.",
        "measurement": "Measured totals exclude sample_warmup_s and use rows_per_indicator=6 full Cartesian workload.",
    }


def benchmark_summary_markdown(payload: dict) -> str:
    """Render benchmark methodology, run summary, stage metrics, and correctness evidence."""
    methodology = payload["methodology"]
    period = payload.get("request", {}).get("period", methodology.get("period", {}))
    lines = [
        "# BTCUSDT 15m Backtest Engine Benchmark",
        "",
        "## Methodology",
        "",
        f"- host: `{methodology['host']}`",
        f"- period_start: `{period['start']}`",
        f"- period_end_exclusive: `{period['end']}`",
        f"- period_semantics: `{period['semantics']}` by 15m `open_time`",
        f"- rows_per_indicator: `{payload['rows_per_indicator']}`",
        f"- warmup_rows_per_indicator: `{payload['warmup_rows_per_indicator']}`",
        "- arities: `1..7`",
        "- risk modes: `none`, `tp_sl_grid`",
        "- direction modes: `long_only`, `long_short_reversal`",
        "- indicator set: `ma.dema`, `ma.hma`, `ma.ema`, `ma.sma`, `ma.wma`, `ma.rma`, `ma.zlema`",
        "- TP/SL grid: `2.0..25.0` inclusive, step `0.5` (`47 x 47 = 2209` cells)",
        "- warmup policy: sample warmup only, no full dry-run warmup; measured totals exclude warmup",
        "- long_only: `+1` opens/holds long; `0` or `-1` closes long; short trades are not opened",
        "- long_short_reversal: `+1` opens/holds long; `-1` opens/holds short; opposite signal closes and reverses",
        "",
        "## Evidence Hashes",
        "",
        f"- request_hash: `{payload['request_hash']}`",
        f"- artifact_manifest_hash: `{payload['artifact_manifest_hash']}`",
        f"- hit_times_manifest_hash: `{payload['hit_times_manifest_hash']}`",
        "",
        "## Run Matrix Summary",
        "",
        "| arity | risk_mode | direction_mode | backend | combos | warmup_combos | sample_warmup_s | exact_s | total_s | peak_rss_mb | cpu_time_s | cpu_pct | top_return_pct | self_check |",
        "|---:|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|",
    ]
    for run in payload["runs"]:
        timers = run["timers"]
        runtime = run["runtime_metrics"]
        top = run["top_results"][0] if run["top_results"] else {}
        lines.append(
            "| {arity} | {risk} | {direction} | `{backend}` | {combos} | {warmup_combos} | {warmup:.6f} | {exact:.6f} | {total:.6f} | {rss:.1f} | {cpu_time:.3f} | {cpu_pct:.1f} | {ret:.6f} | {check} |".format(
                arity=len(run["indicator_ids"]),
                risk=run.get("risk_mode", "none"),
                direction=run.get("direction_mode", ""),
                backend=run.get("exact_engine", ""),
                combos=run.get("cartesian_combinations", 0),
                warmup_combos=run.get("sample_warmup_combinations", 0),
                warmup=run.get("sample_warmup_s", timers.get("sample_warmup", 0.0)),
                exact=timers.get("tp_sl_exact_scoring", timers.get("exact_scoring", 0.0)),
                total=timers.get("total_without_warmup", timers.get("total", 0.0)),
                rss=runtime.get("peak_rss_mb") or 0.0,
                cpu_time=runtime.get("process_cpu_time_s") or 0.0,
                cpu_pct=runtime.get("process_cpu_percent_equivalent") or 0.0,
                ret=top.get("total_return_pct", 0.0),
                check="pass" if run.get("self_check", {}).get("passed") else "fail",
            )
        )
    lines.extend([
        "",
        "## Stage Metrics",
        "",
        "| arity | risk_mode | direction_mode | prepare_s | build_exact_s | build_proxy_s | combo_iter_s | proxy_filter_s | self_check_s | exact_s | heap_s | proxy_fill_s | load_hit_times_s | tp_sl_validation_s | total_s |",
        "|---:|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|",
    ])
    for run in payload["runs"]:
        timers = run["timers"]
        lines.append(
            "| {arity} | {risk} | {direction} | {prepare:.6f} | {build_exact:.6f} | {build_proxy:.6f} | {combo_iter:.6f} | {proxy_filter:.6f} | {self_check:.6f} | {exact:.6f} | {heap:.6f} | {proxy_fill:.6f} | {load_hit:.6f} | {validation:.6f} | {total:.6f} |".format(
                arity=len(run["indicator_ids"]),
                risk=run.get("risk_mode", "none"),
                direction=run.get("direction_mode", ""),
                prepare=timers.get("prepare_pools", 0.0),
                build_exact=timers.get("build_exact_context", 0.0),
                build_proxy=timers.get("build_proxy_context", 0.0),
                combo_iter=timers.get("combo_iteration", 0.0),
                proxy_filter=timers.get("proxy_filter", 0.0),
                self_check=timers.get("self_check", 0.0),
                exact=timers.get("tp_sl_exact_scoring", timers.get("exact_scoring", 0.0)),
                heap=timers.get("heap_update", 0.0),
                proxy_fill=timers.get("top_result_proxy_fill", 0.0),
                load_hit=timers.get("load_hit_times", 0.0),
                validation=timers.get("tp_sl_grid_validation", 0.0),
                total=timers.get("total_without_warmup", timers.get("total", 0.0)),
            )
        )
    lines.extend([
        "",
        "## Correctness And Hashes",
        "",
        "| arity | risk_mode | direction_mode | self_check | trade_count_equal | max_return_diff | best_cell_equal | result_hash |",
        "|---:|---|---|---|---|---:|---|---|",
    ])
    for run in payload["runs"]:
        check = run.get("self_check") or {}
        max_diff = check.get("max_abs_tp_sl_ret_diff", check.get("max_abs_exact_backend_ret_diff", 0.0))
        best_cell = check.get("best_cell_equal", "n/a")
        lines.append(
            "| {arity} | {risk} | {direction} | {passed} | {trade_equal} | {max_diff:.12g} | {best_cell} | `{result_hash}` |".format(
                arity=len(run["indicator_ids"]),
                risk=run.get("risk_mode", "none"),
                direction=run.get("direction_mode", ""),
                passed="pass" if check.get("passed") else "fail",
                trade_equal=check.get("trade_count_equal", False),
                max_diff=float(max_diff or 0.0),
                best_cell=best_cell,
                result_hash=run.get("result_hash", ""),
            )
        )
    lines.extend([
        "",
        "## Sizing Smoke",
        "",
        "| sizing | profit_lock | trade_count | total_return_pct | compiled_parity |",
        "|---|---:|---:|---:|---|",
    ])
    for item in payload.get("sizing_smoke", []):
        parity = item.get("compiled_parity")
        lines.append(
            "| `{sizing}` | {lock} | {trades} | {ret:.6f} | {parity} |".format(
                sizing=item["sizing"]["mode"],
                lock=str(item["profit_lock"]).lower(),
                trades=item["trade_count"],
                ret=item["total_return_pct"],
                parity="pass" if parity is not None else "reference-only",
            )
        )
    return "\n".join(lines) + "\n"


def run_benchmark_matrix(*, rows_per_indicator: int = BENCHMARK_ROWS_PER_INDICATOR, top_k: int = 5, self_check_n: int = 2, output_root: Path = BENCHMARK_OUTPUT_ROOT):
    """Run the required arity x risk x direction benchmark matrix and write evidence files."""
    warmup_rows_per_indicator = min(2, int(rows_per_indicator))
    risk_context = load_tp_sl_hit_times_15m(
        tp_values_pct=REQUEST_RISK_TP_SL_GRID["tp_values_pct"],
        sl_values_pct=REQUEST_RISK_TP_SL_GRID["sl_values_pct"],
    )
    benchmark_runs = []
    for direction_mode in DIRECTION_MODES:
        for arity in INDICATOR_ARITIES:
            indicator_ids = INDICATOR_ARITY_SETS[arity]

            warm_start = time.perf_counter()
            warm_result = run_no_risk_benchmark_once(
                indicator_ids=indicator_ids,
                rows_per_indicator=warmup_rows_per_indicator,
                top_k=1,
                self_check_n=min(1, self_check_n),
                direction_mode=direction_mode,
            )
            no_risk_warm = time.perf_counter() - warm_start
            no_risk = run_no_risk_benchmark_once(
                indicator_ids=indicator_ids,
                rows_per_indicator=rows_per_indicator,
                top_k=top_k,
                self_check_n=self_check_n,
                direction_mode=direction_mode,
            )
            no_risk["risk_mode"] = "none"
            no_risk["sample_warmup_s"] = no_risk_warm
            no_risk["sample_warmup_combinations"] = warm_result["cartesian_combinations"]
            no_risk["service_warmup_s"] = no_risk_warm
            no_risk["numba_warmup_s"] = no_risk_warm
            no_risk["timers"]["sample_warmup"] = no_risk_warm
            no_risk["timers"]["service_warmup"] = no_risk_warm
            no_risk["timers"]["numba_warmup"] = no_risk_warm
            benchmark_runs.append(no_risk)

            warm_start = time.perf_counter()
            warm_risk = run_tp_sl_benchmark_once(
                indicator_ids=indicator_ids,
                rows_per_indicator=warmup_rows_per_indicator,
                top_k=1,
                self_check_n=min(1, self_check_n),
                risk_context=risk_context,
                direction_mode=direction_mode,
            )
            risk_warm = time.perf_counter() - warm_start
            risk_run = run_tp_sl_benchmark_once(
                indicator_ids=indicator_ids,
                rows_per_indicator=rows_per_indicator,
                top_k=top_k,
                self_check_n=self_check_n,
                risk_context=risk_context,
                direction_mode=direction_mode,
            )
            risk_run["sample_warmup_s"] = risk_warm
            risk_run["sample_warmup_combinations"] = warm_risk["cartesian_combinations"]
            risk_run["service_warmup_s"] = risk_warm
            risk_run["numba_warmup_s"] = risk_warm
            risk_run["timers"]["sample_warmup"] = risk_warm
            risk_run["timers"]["service_warmup"] = risk_warm
            risk_run["timers"]["numba_warmup"] = risk_warm
            benchmark_runs.append(risk_run)

    artifact_manifest_hash = hashlib.sha256(SLOT_MANIFEST_PATH.read_bytes()).hexdigest()
    sizing_smoke = run_sizing_smoke()
    methodology = benchmark_methodology_payload(
        rows_per_indicator=rows_per_indicator,
        warmup_rows_per_indicator=warmup_rows_per_indicator,
    )
    payload = {
        "request": BACKTEST_REQUEST,
        "request_hash": REQUEST_HASH,
        "artifact_root": str(ARTIFACT_ROOT),
        "artifact_manifest_hash": artifact_manifest_hash,
        "hit_times_manifest_hash": risk_context["manifest_hash"],
        "rows_per_indicator": rows_per_indicator,
        "warmup_rows_per_indicator": warmup_rows_per_indicator,
        "direction_modes": list(DIRECTION_MODES),
        "methodology": methodology,
        "backend_registry": BACKEND_REGISTRY,
        "sizing_smoke": sizing_smoke,
        "runs": benchmark_runs,
    }
    day = datetime.now(timezone.utc).date().isoformat()
    output_dir = output_root / f"{day}_engine_test_btcusdt_15m"
    output_dir.mkdir(parents=True, exist_ok=True)
    json_path = output_dir / "benchmark_results.json"
    md_path = output_dir / "benchmark_summary.md"
    serializable_payload = json.loads(json.dumps(payload, default=str, allow_nan=True))
    json_path.write_text(json.dumps(serializable_payload, indent=2, sort_keys=True), encoding="utf-8")
    md_path.write_text(benchmark_summary_markdown(serializable_payload), encoding="utf-8")
    payload["output_dir"] = str(output_dir)
    payload["json_path"] = str(json_path)
    payload["markdown_path"] = str(md_path)
    return payload


def resolve_sizing_quote_amount_reference(mode: dict, *, available_quote: float, safe_quote: float, init_cash_quote: float) -> float:
    """Resolve quote amount for smoke-level sizing semantics."""
    equity = available_quote + safe_quote
    mode_name = mode["mode"]
    if mode_name == "all_in":
        quote_amount = available_quote
    elif mode_name == "fixed_quote":
        quote_amount = float(mode["fixed_quote"])
    elif mode_name == "fixed_equity_pct":
        quote_amount = equity * (float(mode["pct"]) / 100.0)
    elif mode_name == "fixed_equity_pct_min_quote":
        quote_amount = max(equity * (float(mode["pct"]) / 100.0), float(mode["min_quote"]))
    elif mode_name == "fixed_equity_pct_max_quote":
        quote_amount = min(equity * (float(mode["pct"]) / 100.0), float(mode["max_quote"]))
    else:
        raise ValueError(f"Unsupported sizing mode: {mode_name!r}")
    return max(0.0, min(float(available_quote), float(quote_amount)))


def score_trade_list_no_risk_sizing_reference(entry_arr: np.ndarray, dir_arr: np.ndarray, exit_arr: np.ndarray, *, sizing: dict, profit_lock: bool) -> dict:
    """Python smoke scorer for production sizing modes."""
    available_quote = float(INIT_CASH_QUOTE)
    safe_quote = 0.0
    equity = float(INIT_CASH_QUOTE)
    closed_trade_count = 0
    for trade_index in range(int(entry_arr.size)):
        entry_idx = int(entry_arr[trade_index])
        if entry_idx >= int(T_exec_limit_1m):
            continue
        exit_idx = int(exit_arr[trade_index])
        if exit_idx < int(T_exec_limit_1m):
            exit_exec_idx = exit_idx
            exit_price_raw = float(price_fields_1m["open"][exit_exec_idx])
        elif CLOSE_ON_END == 1 and int(T_exec_limit_1m) > 0:
            exit_exec_idx = int(T_exec_limit_1m) - 1
            exit_price_raw = float(price_fields_1m["close"][exit_exec_idx])
        else:
            continue
        quote_amount = resolve_sizing_quote_amount_reference(
            sizing,
            available_quote=available_quote,
            safe_quote=safe_quote,
            init_cash_quote=float(INIT_CASH_QUOTE),
        )
        if quote_amount <= 0.0:
            continue
        trade_direction = int(dir_arr[trade_index])
        entry_price_raw = float(price_fields_1m["open"][entry_idx])
        if trade_direction == 1:
            entry_fill_price = entry_price_raw * (1.0 + SLIPPAGE_RATE)
            exit_fill_price = exit_price_raw * (1.0 - SLIPPAGE_RATE)
        else:
            entry_fill_price = entry_price_raw * (1.0 - SLIPPAGE_RATE)
            exit_fill_price = exit_price_raw * (1.0 + SLIPPAGE_RATE)
        qty_base = quote_amount / entry_fill_price
        entry_fee_quote = quote_amount * FEE_RATE
        available_quote -= quote_amount + entry_fee_quote
        exit_quote_amount = qty_base * exit_fill_price
        exit_fee_quote = exit_quote_amount * FEE_RATE
        if trade_direction == 1:
            gross_pnl_quote = exit_quote_amount - quote_amount
        else:
            gross_pnl_quote = quote_amount - exit_quote_amount
        available_quote += quote_amount + gross_pnl_quote - exit_fee_quote
        net_pnl_quote = gross_pnl_quote - entry_fee_quote - exit_fee_quote
        if profit_lock and net_pnl_quote > 0.0:
            locked_profit_quote = net_pnl_quote * (SAFE_PROFIT_PERCENT / 100.0)
            available_quote -= locked_profit_quote
            safe_quote += locked_profit_quote
        equity = available_quote + safe_quote
        closed_trade_count += 1
    return {
        "total_return_pct": ((equity / float(INIT_CASH_QUOTE)) - 1.0) * 100.0,
        "trade_count": closed_trade_count,
        "ending_equity": equity,
        "safe_quote": safe_quote,
    }


def run_sizing_smoke() -> list[dict]:
    """Smoke production sizing modes with profit_lock off/on on a bounded reference trade list."""
    modes = [
        {"mode": "all_in"},
        {"mode": "fixed_quote", "fixed_quote": 100.0},
        {"mode": "fixed_equity_pct", "pct": 10.0},
        {"mode": "fixed_equity_pct_min_quote", "pct": 10.0, "min_quote": 50.0},
        {"mode": "fixed_equity_pct_max_quote", "pct": 10.0, "max_quote": 500.0},
    ]
    indicator_ids = INDICATOR_ARITY_SETS[2]
    row_pools = top_rows_for_request(indicator_ids=indicator_ids, rows_per_indicator=2)
    pools = prepare_indicator_pools(
        indicator_ids=indicator_ids,
        row_pools=row_pools,
        top_frac=1.0,
        min_nonzero=PREFILTER_MIN_NONZERO,
        fee_rate=FEE_RATE,
        time_chunk=TIME_CHUNK,
    )
    entry_arr, dir_arr, exit_arr = build_trade_list_for_indicator_rows_slow(
        indicator_ids=indicator_ids,
        indicator_pools=pools,
        local_indices=(0, 0),
    )
    out = []
    for mode in modes:
        for profit_lock in (False, True):
            reference_metrics = score_trade_list_no_risk_sizing_reference(
                entry_arr,
                dir_arr,
                exit_arr,
                sizing=mode,
                profit_lock=profit_lock,
            )
            compiled_parity = None
            if mode["mode"] in ("all_in", "fixed_quote"):
                metrics = score_trade_list_no_risk(
                    entry_arr,
                    dir_arr,
                    exit_arr,
                    np.int32(entry_arr.size),
                    price_fields_1m["open"],
                    price_fields_1m["close"],
                    T_exec_limit_1m,
                    INIT_CASH_QUOTE,
                    float(mode.get("fixed_quote", FIXED_QUOTE)),
                    FEE_RATE,
                    SLIPPAGE_RATE,
                    SAFE_PROFIT_PERCENT,
                    np.int8(1 if mode["mode"] == "fixed_quote" else 0),
                    np.int8(1 if profit_lock else 0),
                    BARS_PER_YEAR_EXEC_1M,
                    CLOSE_ON_END,
                )
                ret_diff = abs(float(metrics[0]) - float(reference_metrics["total_return_pct"]))
                trade_equal = int(metrics[4]) == int(reference_metrics["trade_count"])
                if ret_diff > 1e-9 or not trade_equal:
                    raise AssertionError(f"Sizing smoke compiled/reference mismatch for {mode['mode']} profit_lock={profit_lock}.")
                compiled_parity = {"return_diff_pct": ret_diff, "trade_count_equal": trade_equal}
            out.append({
                "sizing": mode,
                "profit_lock": bool(profit_lock),
                "passed": True,
                "reference_scorer": "python_sizing_smoke",
                "compiled_parity": compiled_parity,
                **reference_metrics,
            })
    return out



In [ ]:
# Benchmark matrix entry point.
# Override BACKTEST_ENGINE_BENCHMARK_ROWS in the environment for wider or narrower evidence runs.
benchmark_payload = run_benchmark_matrix(
    rows_per_indicator=BENCHMARK_ROWS_PER_INDICATOR,
    top_k=5,
    self_check_n=2,
)
sizing_smoke = benchmark_payload["sizing_smoke"]
print("benchmark output:", benchmark_payload["json_path"])
print("benchmark summary:", benchmark_payload["markdown_path"])
print("sizing smoke cases:", len(sizing_smoke))


In [ ]:
[
    {
        "arity": len(run["indicator_ids"]),
        "risk_mode": run.get("risk_mode", "none"),
        "backend": run.get("exact_engine"),
        "combos": run.get("cartesian_combinations"),
        "exact_s": run["timers"].get("tp_sl_exact_scoring", run["timers"].get("exact_scoring")),
        "total_s": run["timers"].get("total_without_warmup", run["timers"].get("total")),
        "top_return_pct": run["top_results"][0]["total_return_pct"] if run["top_results"] else None,
        "self_check": run.get("self_check"),
    }
    for run in benchmark_payload["runs"]
]


## Production Prototype Acceptance

This notebook is considered valid only when:

- all 1..7 indicator no-risk benchmark runs complete;
- all 1..7 indicator TP/SL risk-on benchmark runs complete;
- TP/SL grid is `2.0..25.0` inclusive with `0.5` step;
- parity/self-check passes for every supported arity and risk mode;
- benchmark JSON and summary markdown are written;
- warmup and runtime metrics include wall time, CPU, memory, Numba threads and artifact/request hashes.
